# Implementation Framework — Reordered Execution Notebook

This notebook reorganizes the original blocks into the correct execution order for the **core IMDb methodology pipeline** and separates the **optional MongoDB validation blocks**.

**Note:** legacy references to `V25`/`mongo_payload_m1` were not included in the main execution flow because the corresponding generator block is not present in the provided notebook.

## Part A — Core methodology pipeline

Run the following cells in order. They create the conceptual scenario, compute the analytical matrix, infer activation classes, and build the experiment catalog and execution template.

### Notes from the original notebook

- `V0` to `V5`: structural logic kept.
- Blocks `1`, `2`, `3` were adapted for the real IMDb scenario via DuckDB.
- `V18` and later blocks handle sharedness and the final analytical matrix.
- The main benchmark-planning output is produced in `V21A`, `V21B`, `V22`, `V22B`, `V22C`, and `V22D`.

In [ ]:
# =========================================================
# BLOCK 1 — REAL IMDb BUNDLE VIA DUCKDB
# =========================================================

from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd

@dataclass
class IMDbDuckDBBundle:
    """
    Main bundle for the real IMDb scenario.

    Instead of storing synthetic pandas tables, this bundle stores:
    - metadados do dataset
    - conexão DuckDB
    - mapeamento das views brutas dos 7 TSVs
    - mapeamento das views conceituais derivadas para o framework
    """
    dataset_name: str
    scale_label: str
    sf_dir: Path
    con: Any
    raw_views: Dict[str, str] = field(default_factory=dict)
    semantic_views: Dict[str, str] = field(default_factory=dict)

    def view_names(self) -> List[str]:
        return sorted(list(self.raw_views.keys()) + list(self.semantic_views.keys()))

    def summary(self) -> pd.DataFrame:
        rows = []

        for scope, mapping in [
            ("raw", self.raw_views),
            ("semantic", self.semantic_views)
        ]:
            for alias, view_name in mapping.items():
                try:
                    desc_df = self.con.execute(f"DESCRIBE {view_name}").df()
                    columns = desc_df["column_name"].tolist()
                    n_rows = self.con.execute(f"SELECT COUNT(*) AS n FROM {view_name}").fetchone()[0]
                except Exception as e:
                    columns = [f"ERRO: {e}"]
                    n_rows = None

                rows.append({
                    "scope": scope,
                    "alias": alias,
                    "view_name": view_name,
                    "rows": n_rows,
                    "columns": columns
                })

        if not rows:
            return pd.DataFrame(columns=["scope", "alias", "view_name", "rows", "columns"])

        return pd.DataFrame(rows).sort_values(
            ["scope", "alias"]
        ).reset_index(drop=True)

    def preview(self, alias: str, limit: int = 5) -> pd.DataFrame:
        """
        Mostra amostra de uma view pelo alias lógico.
        Primeiro procura em semantic_views; se não achar, procura em raw_views.
        """
        if alias in self.semantic_views:
            view_name = self.semantic_views[alias]
        elif alias in self.raw_views:
            view_name = self.raw_views[alias]
        else:
            raise KeyError(
                f"Alias '{alias}' não encontrado. "
                f"Aliases disponíveis: {self.view_names()}"
            )

        return self.con.execute(
            f"SELECT * FROM {view_name} LIMIT {int(limit)}"
        ).df()

In [ ]:
# =========================================================
# BLOCK 2 — REGISTER THE 7 SCALE-FACTOR TSVs IN DUCKDB
# =========================================================

from pathlib import Path
import importlib.util
import subprocess
import sys

# ---------------------------------------------------------
# 1) Ensure duckdb is installed
# ---------------------------------------------------------
if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])

import duckdb

# ---------------------------------------------------------
# 2) Ajuste os caminhos dos scale factors
# Cada pasta deve conter os 7 arquivos TSV exportados
# ---------------------------------------------------------
IMDB_SF_PATHS = {
    "sf0.25": Path("/home/jovyan/privado/imdb/sf_outputs/sf_025"),
    "sf0.5":  Path("/home/jovyan/privado/imdb/sf_outputs/sf_050"),
    "sf1":    Path("/home/jovyan/privado/imdb/sf_outputs/sf_1"),
}

# Escolha o SF ativo para esta execução
ACTIVE_SCALE_LABEL = "sf0.25"

if ACTIVE_SCALE_LABEL not in IMDB_SF_PATHS:
    raise KeyError(
        f"Scale factor '{ACTIVE_SCALE_LABEL}' não encontrado em IMDB_SF_PATHS."
    )

sf_dir = IMDB_SF_PATHS[ACTIVE_SCALE_LABEL]

if not sf_dir.exists():
    raise FileNotFoundError(
        f"Pasta do scale factor não encontrada: {sf_dir}\n"
        "Ajuste os caminhos em IMDB_SF_PATHS antes de continuar."
    )

# ---------------------------------------------------------
# 3) Create conexão DuckDB em memória
# ---------------------------------------------------------
con = duckdb.connect(database=":memory:")

# ---------------------------------------------------------
# 4) Schemas explícitos dos 7 TSVs
# Tudo como VARCHAR para seguir a metodologia de parsing estável
# ---------------------------------------------------------
TSV_SCHEMAS = {
    "name_basics": {
        "filename": "name.basics.tsv",
        "columns": {
            "nconst": "VARCHAR",
            "primaryName": "VARCHAR",
            "birthYear": "VARCHAR",
            "deathYear": "VARCHAR",
            "primaryProfession": "VARCHAR",
            "knownForTitles": "VARCHAR",
        },
        "max_line_size": 1_000_000,
    },
    "title_akas": {
        "filename": "title.akas.tsv",
        "columns": {
            "titleId": "VARCHAR",
            "ordering": "VARCHAR",
            "title": "VARCHAR",
            "region": "VARCHAR",
            "language": "VARCHAR",
            "types": "VARCHAR",
            "attributes": "VARCHAR",
            "isOriginalTitle": "VARCHAR",
        },
        "max_line_size": 2_000_000,
    },
    "title_basics": {
        "filename": "title.basics.tsv",
        "columns": {
            "tconst": "VARCHAR",
            "titleType": "VARCHAR",
            "primaryTitle": "VARCHAR",
            "originalTitle": "VARCHAR",
            "isAdult": "VARCHAR",
            "startYear": "VARCHAR",
            "endYear": "VARCHAR",
            "runtimeMinutes": "VARCHAR",
            "genres": "VARCHAR",
        },
        "max_line_size": 1_000_000,
    },
    "title_crew": {
        "filename": "title.crew.tsv",
        "columns": {
            "tconst": "VARCHAR",
            "directors": "VARCHAR",
            "writers": "VARCHAR",
        },
        "max_line_size": 1_000_000,
    },
    "title_episode": {
        "filename": "title.episode.tsv",
        "columns": {
            "tconst": "VARCHAR",
            "parentTconst": "VARCHAR",
            "seasonNumber": "VARCHAR",
            "episodeNumber": "VARCHAR",
        },
        "max_line_size": 1_000_000,
    },
    "title_principals": {
        "filename": "title.principals.tsv",
        "columns": {
            "tconst": "VARCHAR",
            "ordering": "VARCHAR",
            "nconst": "VARCHAR",
            "category": "VARCHAR",
            "job": "VARCHAR",
            "characters": "VARCHAR",
        },
        "max_line_size": 2_000_000,
    },
    "title_ratings": {
        "filename": "title.ratings.tsv",
        "columns": {
            "tconst": "VARCHAR",
            "averageRating": "VARCHAR",
            "numVotes": "VARCHAR",
        },
        "max_line_size": 1_000_000,
    },
}

# ---------------------------------------------------------
# 5) Funções auxiliares
# ---------------------------------------------------------
def duckdb_columns_map_sql(columns: dict) -> str:
    return "{" + ", ".join(f"'{k}': '{v}'" for k, v in columns.items()) + "}"

def create_tsv_view(con, view_name: str, file_path: Path, columns: dict, max_line_size: int):
    file_sql = str(file_path).replace("\\", "/").replace("'", "''")
    columns_sql = duckdb_columns_map_sql(columns)

    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv(
        '{file_sql}',
        delim='\\t',
        header=true,
        columns={columns_sql},
        nullstr='\\N',
        auto_detect=false,
        parallel=false,
        quote='',
        escape='',
        max_line_size={max_line_size}
    );
    """
    con.execute(sql)

# ---------------------------------------------------------
# 6) Registrar os 7 arquivos como views
# ---------------------------------------------------------
raw_views = {}

for alias, spec in TSV_SCHEMAS.items():
    file_path = sf_dir / spec["filename"]

    if not file_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")

    view_name = f"imdb_{alias}"

    create_tsv_view(
        con=con,
        view_name=view_name,
        file_path=file_path,
        columns=spec["columns"],
        max_line_size=spec["max_line_size"],
    )

    raw_views[alias] = view_name

# ---------------------------------------------------------
# 7) Create o bundle principal
# ---------------------------------------------------------
imdb_data_bundle = IMDbDuckDBBundle(
    dataset_name="IMDb",
    scale_label=ACTIVE_SCALE_LABEL,
    sf_dir=sf_dir,
    con=con,
    raw_views=raw_views,
    semantic_views={}
)

print("Scale factor ativo:", imdb_data_bundle.scale_label)
print("Pasta:", imdb_data_bundle.sf_dir)
print("\nViews brutas registradas:")
for k, v in imdb_data_bundle.raw_views.items():
    print(f"- {k} -> {v}")

print("\nSummary inicial:")
display(imdb_data_bundle.summary())

In [ ]:
# =========================================================
# BLOCK 3 — DERIVED CONCEPTUAL VIEWS FOR IMDb
# =========================================================

required_names = ["imdb_data_bundle"]
for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o BLOCK 2."
        )

con = imdb_data_bundle.con

# ---------------------------------------------------------
# 1) WatchItem = general root entity for titles
#    Main base: title.basics
#    Enriquecimento opcional: title.ratings
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_watchitems AS
SELECT
    b.tconst AS watchitem_id,
    b.titleType AS title_type,
    b.primaryTitle AS title,
    b.originalTitle AS original_title,
    b.isAdult AS is_adult,
    b.startYear AS release_year,
    b.endYear AS end_year,
    b.runtimeMinutes AS runtime_minutes,
    b.genres AS genres_raw,
    CASE
        WHEN b.genres IS NULL THEN NULL
        ELSE split_part(b.genres, ',', 1)
    END AS primary_genre,
    r.averageRating AS avg_rating,
    r.numVotes AS num_votes
FROM imdb_title_basics b
LEFT JOIN imdb_title_ratings r
    ON b.tconst = r.tconst
""")

# ---------------------------------------------------------
# 2) Person = pessoas do domínio
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_persons AS
SELECT
    nconst AS person_id,
    primaryName AS person_name,
    birthYear AS birth_year,
    deathYear AS death_year,
    primaryProfession AS primary_profession,
    knownForTitles AS known_for_titles
FROM imdb_name_basics
""")

# ---------------------------------------------------------
# 3) Role = entidade associativa entre WatchItem e Person
#    Cada linha de title.principals vira um Role
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_roles AS
SELECT
    tconst || '#' || ordering AS role_id,
    tconst AS watchitem_id,
    nconst AS person_id,
    ordering AS principal_order,
    category AS role_category,
    job,
    characters
FROM imdb_title_principals
""")

# ---------------------------------------------------------
# 4) Genre = descritor derivado
#    Aqui usamos apenas o gênero principal para manter
#    compatibilidade com a metodologia atual do notebook
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_genres AS
SELECT DISTINCT
    primary_genre AS genre_name
FROM imdb_watchitems
WHERE primary_genre IS NOT NULL
""")

# ---------------------------------------------------------
# 5) Movie = subtipo de WatchItem
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_movies AS
SELECT
    watchitem_id,
    title,
    original_title,
    release_year,
    end_year,
    runtime_minutes,
    primary_genre,
    avg_rating,
    num_votes
FROM imdb_watchitems
WHERE title_type = 'movie'
""")

# ---------------------------------------------------------
# 6) Series = subtipo de WatchItem
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_series AS
SELECT
    watchitem_id,
    title,
    original_title,
    release_year,
    end_year,
    runtime_minutes,
    primary_genre,
    avg_rating,
    num_votes
FROM imdb_watchitems
WHERE title_type = 'tvSeries'
""")

# ---------------------------------------------------------
# 7) Episode = componente ligado a Series
#    Aqui usamos title.episode como ligação explícita
# ---------------------------------------------------------
con.execute("""
CREATE OR REPLACE VIEW imdb_episodes AS
SELECT
    e.tconst AS episode_watchitem_id,
    e.parentTconst AS series_watchitem_id,
    e.seasonNumber AS season_number,
    e.episodeNumber AS episode_number
FROM imdb_title_episode e
""")

# ---------------------------------------------------------
# 8) Atualizar o bundle com as views conceituais
# ---------------------------------------------------------
imdb_data_bundle.semantic_views = {
    "watchitems": "imdb_watchitems",
    "persons": "imdb_persons",
    "roles": "imdb_roles",
    "genres": "imdb_genres",
    "movies": "imdb_movies",
    "series": "imdb_series",
    "episodes": "imdb_episodes",
}

print("Views conceituais registradas:")
for k, v in imdb_data_bundle.semantic_views.items():
    print(f"- {k} -> {v}")

print("\nSummary atualizado do bundle:")
display(imdb_data_bundle.summary())

print("\nPrévia: watchitems")
display(imdb_data_bundle.preview("watchitems", 5))

print("\nPrévia: roles")
display(imdb_data_bundle.preview("roles", 5))

print("\nPrévia: episodes")
display(imdb_data_bundle.preview("episodes", 5))

In [ ]:
# =========================================================
# INSTANCE 1 — SCENARIO IMDb REAL
# full methodological workload QG1–QG10
# without User, focused on the content fragment
# =========================================================

required_names = ["imdb_data_bundle"]
for name in required_names:
    if name not in globals():
        raise NameError(f"'{name}' is not defined. Run first os blocos 1, 2 e 3.")

# ---------------------------------------------------------
# 1) Conceptual schema of the content fragment
# ---------------------------------------------------------
imdb_scenario = ScenarioSpec(
    name="IMDb_Content_QG1_QG10",
    description=(
        "Cenário conceitual do IMDb real (SF0.25/SF0.5/SF1) focado "
        "no fragmento de conteúdo, com workload metodológico completo "
        "instanciando QG1–QG10."
    ),
    entities=[
        "Person",
        "Genre",
        "Role",
        "WatchItem",
        "Series",
        "Movie",
        "Episode",
    ],
    relationships=[
        # descritor compartilhado
        {"source": "Genre",   "target": "WatchItem", "name": "has_genre"},

        # subtipos estruturais de WatchItem
        {"source": "Movie",   "target": "WatchItem", "name": "movie_is_watchitem"},
        {"source": "Series",  "target": "WatchItem", "name": "series_is_watchitem"},
        {"source": "Episode", "target": "WatchItem", "name": "episode_is_watchitem"},

        # caminho associativo principal
        {"source": "Person",  "target": "Role",      "name": "person_in_role"},
        {"source": "Role",    "target": "WatchItem", "name": "role_of_watchitem"},

        # containment explícito
        {"source": "Series",  "target": "Episode",   "name": "series_contains_episode"},
    ],
    workload_queries=[
        # -------------------------------------------------
        # QG1 — busca pontual / identidade
        # -------------------------------------------------
        {
            "name": "QG1_WatchItemById",
            "type": "select",
            "generic_class": "QG1",
            "entities": ["WatchItem"],
            "abstract_query": "RETURN WatchItem WHERE watchitem_id = ?",
            "notes": "Busca pontual por identificador único de WatchItem."
        },

        # -------------------------------------------------
        # QG2 — recuperação simples de entidade
        # -------------------------------------------------
        {
            "name": "QG2_WatchItemByTitle",
            "type": "select",
            "generic_class": "QG2",
            "entities": ["WatchItem"],
            "abstract_query": "RETURN ALL FROM WatchItem WHERE title = ?",
            "notes": "Recuperação simples da entidade raiz por atributo direto."
        },

        # -------------------------------------------------
        # QG3 — recuperação filtrada centrada na raiz
        # -------------------------------------------------
        {
            "name": "QG3_RecommendationByGenreAndSubtype",
            "type": "select",
            "generic_class": "QG3",
            "entities": ["WatchItem", "Genre", "Movie", "Series", "Episode"],
            "abstract_query": (
                "RETURN WatchItem recommendations "
                "FILTER BY genre, subtype and nearby title components"
            ),
            "notes": (
                "Query centrada em WatchItem com filtros sobre descritor "
                "compartilhado e subtipos/componentes próximos."
            )
        },

        # -------------------------------------------------
        # QG4 — recuperação associativa
        # -------------------------------------------------
        {
            "name": "QG4_AllPersonsOfTypeForWatchItem",
            "type": "select",
            "generic_class": "QG4",
            "entities": ["WatchItem", "Role", "Person"],
            "abstract_query": (
                "RETURN Person.ALL FROM WatchItem "
                "WHERE watchitem_id = ? AND role_category = ?"
            ),
            "notes": (
                "Consulta associativa que parte de WatchItem, passa por Role "
                "e alcança Person."
            )
        },

        # -------------------------------------------------
        # QG5 — travessia profunda / multi-hop
        # -------------------------------------------------
        {
            "name": "QG5_AllPersonsForEpisodesOfSeries",
            "type": "select",
            "generic_class": "QG5",
            "entities": ["Series", "Episode", "WatchItem", "Role", "Person"],
            "abstract_query": (
                "RETURN Person.ALL FOR all Episode of Series "
                "WHERE series_watchitem_id = ?"
            ),
            "notes": (
                "Travessia multi-hop que percorre Series -> Episode -> WatchItem "
                "-> Role -> Person."
            )
        },

        # -------------------------------------------------
        # QG6 — containment / composição
        # -------------------------------------------------
        {
            "name": "QG6_EpisodesOfSeries",
            "type": "select",
            "generic_class": "QG6",
            "entities": ["Series", "Episode"],
            "abstract_query": (
                "RETURN Episode.ALL FROM Series "
                "WHERE series_watchitem_id = ?"
            ),
            "notes": (
                "Query de containment para a relação parte-todo Series -> Episode."
            )
        },

        # -------------------------------------------------
        # QG7 — inserção/atualização isolada
        # -------------------------------------------------
        {
            "name": "QG7_UpdateWatchItemMetadata",
            "type": "update",
            "generic_class": "QG7",
            "entities": ["WatchItem"],
            "abstract_query": (
                "UPDATE WatchItem "
                "SET runtime_minutes = ?, primary_genre = ? "
                "WHERE watchitem_id = ?"
            ),
            "notes": (
                "Escrita isolada sobre a entidade raiz, sem criação "
                "de novo relacionamento."
            )
        },

        # -------------------------------------------------
        # QG8 — inserção/atualização com criação de relacionamento
        # -------------------------------------------------
        {
            "name": "QG8_AddPersonRoleToWatchItem",
            "type": "insert",
            "generic_class": "QG8",
            "entities": ["Person", "WatchItem", "Role"],
            "abstract_query": (
                "INSERT Person if needed and connect Person to WatchItem via Role"
            ),
            "notes": (
                "Escrita com criação de vínculo estrutural via entidade associativa."
            )
        },

        # -------------------------------------------------
        # QG9 — agregação / ranking / top-k
        # -------------------------------------------------
        {
            "name": "QG9_TopRatedSeriesByGenre",
            "type": "select",
            "generic_class": "QG9",
            "entities": ["WatchItem", "Series", "Genre"],
            "abstract_query": (
                "RETURN TOP K Series BY avg_rating "
                "GROUPED/FILTERED BY Genre"
            ),
            "notes": (
                "Consulta analítica com agrupamento/ordenação sobre Series "
                "e descritor de gênero."
            )
        },

        # -------------------------------------------------
        # QG10 — busca com ranking e filtros avançados
        # -------------------------------------------------
        {
            "name": "QG10_AdvancedSearchWatchItems",
            "type": "select",
            "generic_class": "QG10",
            "entities": ["WatchItem", "Genre", "Movie", "Series", "Episode"],
            "abstract_query": (
                "SEARCH WatchItem BY title keywords, genre, year range, "
                "rating threshold and subtype ORDERED BY ranking"
            ),
            "notes": (
                "Busca mais extensa com múltiplos filtros e ordenação/ranking."
            )
        },
    ]
)

# ---------------------------------------------------------
# 2) Fragmento metodológico
# ---------------------------------------------------------
hubara_imdb_recommendation = RecommendationSpec(
    recommendation_name="IMDb_Content_QG1_QG10_Workload",
    source_method="Generic documentary methodology instantiated on IMDb content",
    source_dataset="IMDb",
    fragments=[
        FragmentSpec(
            name="ContentFragment",
            entities=["Episode", "Genre", "Movie", "Person", "Role", "Series", "WatchItem"],
            model="document_or_column_candidate",
            notes=(
                "Fragmento de conteúdo do IMDb usado para extração das variáveis "
                "da matriz, ativação das classes e benchmark futuro."
            )
        )
    ]
)

# ---------------------------------------------------------
# 3) Plano físico-base de referência
# ---------------------------------------------------------
hubara_imdb_mongo_materialization = MaterializationPlan(
    scenario_name="IMDb_Content_QG1_QG10",
    recommendation_name="IMDb_Content_Mongo_Baseline",
    fragment_to_db={
        "ContentFragment": PhysicalDBSpec(
            model="document",
            engine="MongoDB",
            host="127.0.0.1",
            port=57017,
            database_name="imdb_content_mongo_db",
            username="mongo",
            password="mongo"
        ),
    }
)

# ---------------------------------------------------------
# 4) Workload do benchmark
#    Aqui mantemos exatamente as 10 instâncias concretas
#    correspondentes às classes QG1–QG10
# ---------------------------------------------------------
imdb_workload_mongo = WorkloadSpec(
    name="IMDb_Content_Methodology_Workload_QG1_QG10",
    description=(
        "Workload metodológico completo do fragmento de conteúdo do IMDb, "
        "instanciando uma query concreta para cada classe QG1–QG10."
    ),
    queries=[
        BenchmarkQuery(
            name="QG1_WatchItemById",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN WatchItem WHERE watchitem_id = ?",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG2_WatchItemByTitle",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG3_RecommendationByGenreAndSubtype",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query=(
                "RETURN WatchItem recommendations "
                "FILTER BY genre, subtype and nearby title components"
            ),
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG4_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Role", "Person"],
            abstract_query=(
                "RETURN Person.ALL FROM WatchItem "
                "WHERE watchitem_id = ? AND role_category = ?"
            ),
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG5_AllPersonsForEpisodesOfSeries",
            query_type="select",
            entities_involved=["Series", "Episode", "WatchItem", "Role", "Person"],
            abstract_query=(
                "RETURN Person.ALL FOR all Episode of Series "
                "WHERE series_watchitem_id = ?"
            ),
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG6_EpisodesOfSeries",
            query_type="select",
            entities_involved=["Series", "Episode"],
            abstract_query="RETURN Episode.ALL FROM Series WHERE series_watchitem_id = ?",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG7_UpdateWatchItemMetadata",
            query_type="update",
            entities_involved=["WatchItem"],
            abstract_query=(
                "UPDATE WatchItem "
                "SET runtime_minutes = ?, primary_genre = ? "
                "WHERE watchitem_id = ?"
            ),
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG8_AddPersonRoleToWatchItem",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person if needed and connect Person to WatchItem via Role",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG9_TopRatedSeriesByGenre",
            query_type="select",
            entities_involved=["WatchItem", "Series", "Genre"],
            abstract_query="RETURN TOP K Series BY avg_rating GROUPED/FILTERED BY Genre",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="QG10_AdvancedSearchWatchItems",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query=(
                "SEARCH WatchItem BY title keywords, genre, year range, "
                "rating threshold and subtype ORDERED BY ranking"
            ),
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
    ],
    repetitions=10
)

# ---------------------------------------------------------
# 5) Visões auxiliares para inspeção rápida
# ---------------------------------------------------------
imdb_workload_core_df = pd.DataFrame(imdb_scenario.workload_queries)

imdb_workload_benchmark_df = pd.DataFrame([
    {
        "name": q.name,
        "query_type": q.query_type,
        "entities_involved": q.entities_involved,
        "expected_fragment": q.expected_fragment,
        "expected_db_model": q.expected_db_model,
        "abstract_query": q.abstract_query,
    }
    for q in imdb_workload_mongo.queries
])

imdb_query_class_summary_df = imdb_workload_core_df[
    ["name", "generic_class", "type", "entities", "notes"]
].copy()

print("Cenário IMDb com workload QG1–QG10 atualizado com sucesso.")

print("\nEntidades do cenário:")
print(imdb_scenario.entities)

print("\nRelacionamentos do cenário:")
display(pd.DataFrame(imdb_scenario.relationships))

print("\nSummary das queries metodológicas instanciadas:")
display(imdb_query_class_summary_df)

print("\nWorkload do benchmark:")
display(imdb_workload_benchmark_df)

In [ ]:
# =========================================================
# BLOCK V0 — PREPARE THE CONCEPTUAL WORKLOAD
# =========================================================
#INSTANCE R(Qi) = quantity entities touched

import pandas as pd

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "imdb_scenario" not in globals():
    raise NameError(
        "The object 'imdb_scenario' is not defined. "
        "Run first the block that creates the IMDb scenario."
    )

# ---------------------------------------------------------
# 2) Transformar o workload do cenário em DataFrame
# ---------------------------------------------------------
workload_conceptual_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

# Renomeia colunas para deixar mais explícito
workload_conceptual_df = workload_conceptual_df.rename(columns={
    "name": "query_name",
    "type": "query_type",
    "entities": "entities_touched"
})

# Garante uma coluna com quantidade de entidades tocadas
workload_conceptual_df["n_entities_touched"] = workload_conceptual_df["entities_touched"].apply(len)

print("Workload conceitual do IMDb:")
display(workload_conceptual_df)

In [ ]:
# =========================================================
# BLOCK V1 — ORGANIZE THE CONCEPTUAL RELATIONSHIPS
# =========================================================
#Organize the conceptual relationships

from collections import defaultdict

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "imdb_scenario" not in globals():
    raise NameError(
        "The object 'imdb_scenario' is not defined. "
        "Run first the block that creates the IMDb scenario."
    )

# ---------------------------------------------------------
# 2) DataFrame com os relacionamentos conceituais
# ---------------------------------------------------------
conceptual_relationships_df = pd.DataFrame(imdb_scenario.relationships).copy()

# Cria um identificador textual da aresta conceitual
conceptual_relationships_df["edge_id"] = conceptual_relationships_df.apply(
    lambda row: f'{row["source"]} -- {row["target"]} [{row["name"]}]',
    axis=1
)

print("Relacionamentos conceituais do esquema IMDb:")
display(conceptual_relationships_df)

# ---------------------------------------------------------
# 3) Lista de arestas em formato Python
# ---------------------------------------------------------
conceptual_edges = conceptual_relationships_df[
    ["source", "target", "name", "edge_id"]
].to_dict(orient="records")

# ---------------------------------------------------------
# 4) Grafo não-direcionado simples para navegação conceitual
# Note:
# para calcular conectividade conceitual entre entidades,
# vamos tratar os relacionamentos como não-direcionados.
# ---------------------------------------------------------
conceptual_adj = defaultdict(list)

for edge in conceptual_edges:
    src = edge["source"]
    tgt = edge["target"]
    conceptual_adj[src].append((tgt, edge["name"], edge["edge_id"]))
    conceptual_adj[tgt].append((src, edge["name"], edge["edge_id"]))

print("Entidades presentes no grafo conceitual:")
print(sorted(conceptual_adj.keys()))

In [ ]:
# =========================================================
# BLOCK V2 — EXTRACT Rc(Qi) EXACTLY
# =========================================================
# To EXTRACT Rc(Qi) exactily

from itertools import combinations
from collections import deque

# ---------------------------------------------------------
# 1) Helper function:
# checks whether a subset of edges connects all
# terminal entities of a query
# ---------------------------------------------------------
def terminals_are_connected(selected_edges, terminals):
    """
    Retorna True se todas as entidades 'terminals' estiverem
    conectadas no subgrafo formado por 'selected_edges'.
    """
    terminals = list(dict.fromkeys(terminals))  # remove duplicatas preservando ordem

    if len(terminals) <= 1:
        return True

    # monta adjacência apenas com as arestas selecionadas
    adj = defaultdict(set)
    for edge in selected_edges:
        adj[edge["source"]].add(edge["target"])
        adj[edge["target"]].add(edge["source"])

    # busca em largura a partir do primeiro terminal
    start = terminals[0]
    visited = set([start])
    queue = deque([start])

    while queue:
        current = queue.popleft()
        for neighbor in adj[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    # todos os terminais precisam estar no mesmo componente
    return all(t in visited for t in terminals)

# ---------------------------------------------------------
# 2) Função principal:
# calcula o número mínimo de relacionamentos conceituais
# necessários para conectar as entidades da query
# ---------------------------------------------------------
def extract_rc_for_query(terminals, conceptual_edges):
    """
    Retorna um dicionário com:
    - rc: número mínimo de relacionamentos conceituais
    - selected_edge_ids: identificadores das arestas escolhidas
    """
    terminals = list(dict.fromkeys(terminals))

    # Se a query toca 0 ou 1 entidade, não há relacionamento a percorrer
    if len(terminals) <= 1:
        return {
            "rc": 0,
            "selected_edge_ids": []
        }

    # Testa subconjuntos de arestas por tamanho crescente.
    # O primeiro que conectar todos os terminais já é ótimo.
    for k in range(1, len(conceptual_edges) + 1):
        for subset in combinations(conceptual_edges, k):
            nodes_in_subset = set()
            for edge in subset:
                nodes_in_subset.add(edge["source"])
                nodes_in_subset.add(edge["target"])

            # poda simples: se faltam terminais, nem precisa testar conectividade
            if not set(terminals).issubset(nodes_in_subset):
                continue

            if terminals_are_connected(subset, terminals):
                return {
                    "rc": k,
                    "selected_edge_ids": [edge["edge_id"] for edge in subset]
                }

    # Se nada conectar, devolve None (não esperado neste cenário)
    return {
        "rc": None,
        "selected_edge_ids": []
    }

# ---------------------------------------------------------
# 3) Apply para cada query do workload
# ---------------------------------------------------------
rc_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    terminals = row["entities_touched"]

    rc_info = extract_rc_for_query(terminals, conceptual_edges)

    rc_rows.append({
        "query_name": query_name,
        "query_type": row["query_type"],
        "entities_touched": terminals,
        "n_entities_touched": len(terminals),
        "Rc": rc_info["rc"],
        "Rc_selected_edges": rc_info["selected_edge_ids"]
    })

rc_df = pd.DataFrame(rc_rows)

print("Extração de Rc(Qi):")
display(rc_df)

#Rc -->  relationship number  of query
#Rc(Qi, Er, D) --> relationship number in time execution
# ΔR: how much was reduced. (diferrence between RC and Rc(Qi, Er, D)


In [ ]:
# =========================================================
# BLOCK V3 — INITIAL ROOT CANDIDATES er
# =========================================================

from collections import deque

# ---------------------------------------------------------
# 1) Shortest distance between two entities in the conceptual graph
# ---------------------------------------------------------
def shortest_entity_distance(source_entity, target_entity, adj):
    """
    Returns the shortest distance (em número de arestas conceituais)
    between two schema entities.
    """
    if source_entity == target_entity:
        return 0

    visited = set([source_entity])
    queue = deque([(source_entity, 0)])

    while queue:
        current, dist = queue.popleft()

        for neighbor, rel_name, edge_id in adj[current]:
            if neighbor == target_entity:
                return dist + 1

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))

    return None

# ---------------------------------------------------------
# 2) Heurística de score para raiz:
# - menor distância média até as outras entidades da query
# - menor distância máxima
# - maior número de vizinhos diretos entre entidades da query
# ---------------------------------------------------------
def extract_root_candidates(terminals, adj):
    """
    Retorna um DataFrame ranqueando candidatos a raiz.
    """
    terminals = list(dict.fromkeys(terminals))
    rows = []

    for candidate in terminals:
        distances = []
        direct_links = 0

        for other in terminals:
            if other == candidate:
                continue

            d = shortest_entity_distance(candidate, other, adj)
            distances.append(d)

            if d == 1:
                direct_links += 1

        avg_distance = sum(distances) / len(distances) if distances else 0.0
        max_distance = max(distances) if distances else 0

        rows.append({
            "candidate_root": candidate,
            "avg_distance_to_other_touched_entities": avg_distance,
            "max_distance_to_other_touched_entities": max_distance,
            "direct_links_to_other_touched_entities": direct_links
        })

    root_df = pd.DataFrame(rows).sort_values(
        by=[
            "avg_distance_to_other_touched_entities",
            "max_distance_to_other_touched_entities",
            "direct_links_to_other_touched_entities"
        ],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return root_df

# ---------------------------------------------------------
# 3) Apply para cada query
# ---------------------------------------------------------
root_candidates_by_query = {}

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    terminals = row["entities_touched"]

    root_candidates_df = extract_root_candidates(terminals, conceptual_adj)
    root_candidates_by_query[query_name] = root_candidates_df

    print("\n" + "=" * 80)
    print(f"Candidatos iniciais de raiz para {query_name}")
    display(root_candidates_df)

In [ ]:
# =========================================================
# BLOCK V4 — DEFINE THE SELECTED ROOT PER QUERY
# aligned to workload QG1–QG10
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Manual/assisted definition das raízes escolhidas
# ---------------------------------------------------------
# Adopted criterion:
# - use the structural heuristic as support;
# - apply semantic decision quando a raiz documental mais útil
#   para o experimento não coincidir com a entidade mais central.
#
# Note:
# the methodology allows exactly this combination:
# structural heuristic + query-guided semantic decision.

selected_roots_df = pd.DataFrame([
    {
        "query_name": "QG1_WatchItemById",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A query toca apenas WatchItem; WatchItem é a raiz natural."
        )
    },
    {
        "query_name": "QG2_WatchItemByTitle",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A query toca apenas WatchItem; WatchItem é a raiz natural."
        )
    },
    {
        "query_name": "QG3_RecommendationByGenreAndSubtype",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "WatchItem é a entidade central da query e concentra Genre, "
            "Movie, Series e Episode no subgrafo conceitual. "
            "Essa escolha também fica alinhada ao caso clássico de query rasa "
            "centrada em WatchItem."
        )
    },
    {
        "query_name": "QG4_AllPersonsOfTypeForWatchItem",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "Embora a query atravesse o caminho associativo WatchItem -> Role -> Person, "
            "ela é semanticamente centrada no watch item. "
            "Por isso, WatchItem foi mantido como raiz documental."
        )
    },
    {
        "query_name": "QG5_AllPersonsForEpisodesOfSeries",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A query parte semanticamente de Series, mas no subgrafo mínimo "
            "WatchItem funciona como entidade mais central de ligação entre "
            "Series, Episode, Role e Person. "
            "Além disso, manter WatchItem como raiz ajuda a comparar esta query "
            "com a família associativa centrada no agregado principal."
        )
    },
    {
        "query_name": "QG6_EpisodesOfSeries",
        "selected_root": "Series",
        "root_decision_rationale": (
            "A query representa um caso de containment/composição; "
            "Series é a raiz natural do agregado Series -> Episode."
        )
    },
    {
        "query_name": "QG7_UpdateWatchItemMetadata",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A query toca apenas WatchItem; WatchItem é a raiz natural."
        )
    },
    {
        "query_name": "QG8_AddPersonRoleToWatchItem",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "Role é uma entidade associativa muito central estruturalmente, "
            "mas a decisão documental aqui é centrada em WatchItem, porque "
            "o vínculo criado é interpretado como parte do agregado do item."
        )
    },
    {
        "query_name": "QG9_TopRatedSeriesByGenre",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "Embora o ranking seja sobre Series, a estrutura da query passa por "
            "Genre e pelo subtipo Series como componente de WatchItem. "
            "Por isso, WatchItem foi escolhido como raiz principal."
        )
    },
    {
        "query_name": "QG10_AdvancedSearchWatchItems",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A busca avançada continua centrada na entidade raiz WatchItem, "
            "com filtros adicionais sobre descritor e subtipo."
        )
    },
])

print("Raízes documentais provisórias selecionadas:")
display(selected_roots_df)

In [ ]:
# =========================================================
# BLOCK V5 — COMPUTE D(E, er, Qi)
# aligned to workload QG1–QG10
# =========================================================

from collections import defaultdict, deque
import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "conceptual_edges",
    "selected_roots_df",
    "imdb_scenario"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

if "rc_df" not in globals():
    print("Warning: 'rc_df' não encontrado. Queries com Rc > 0 podem ficar incompletas.")

# ---------------------------------------------------------
# 2) Base do workload: usar TODAS as queries do cenário
# ---------------------------------------------------------
workload_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

# normalizar nome da query
if "query_name" not in workload_df.columns:
    if "name" in workload_df.columns:
        workload_df = workload_df.rename(columns={"name": "query_name"})
    else:
        raise ValueError(
            "O workload não tem nem 'query_name' nem 'name'."
        )

# normalizar demais colunas esperadas
if "entities" not in workload_df.columns:
    raise ValueError("workload_df precisa ter a coluna 'entities'.")

if "generic_class" not in workload_df.columns:
    workload_df["generic_class"] = None

if "type" not in workload_df.columns:
    workload_df["type"] = None

if "notes" not in workload_df.columns:
    workload_df["notes"] = None

# ---------------------------------------------------------
# 3) Índice auxiliar: mapear edge_id -> aresta conceitual
# ---------------------------------------------------------
edge_by_id = {
    edge["edge_id"]: edge
    for edge in conceptual_edges
}

# ---------------------------------------------------------
# 4) Funções auxiliares
# ---------------------------------------------------------
def build_subgraph_adjacency_from_edge_ids(edge_ids):
    adj = defaultdict(list)

    for edge_id in edge_ids:
        if edge_id not in edge_by_id:
            continue

        edge = edge_by_id[edge_id]
        src = edge["source"]
        tgt = edge["target"]

        adj[src].append(tgt)
        adj[tgt].append(src)

    return adj


def shortest_distance_in_subgraph(root, target, sub_adj):
    if root == target:
        return 0

    visited = {root}
    queue = deque([(root, 0)])

    while queue:
        current, dist = queue.popleft()

        for neighbor in sub_adj.get(current, []):
            if neighbor == target:
                return dist + 1

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))

    return None


def normalize_edge_list(x):
    if isinstance(x, list):
        return x
    return []

# ---------------------------------------------------------
# 5) Mesclar workload + raízes + Rc
# ---------------------------------------------------------
base_df = workload_df.merge(
    selected_roots_df,
    on="query_name",
    how="left"
)

if "rc_df" in globals():
    rc_subset_cols = [c for c in ["query_name", "Rc", "Rc_selected_edges"] if c in rc_df.columns]
    rc_subset_df = rc_df[rc_subset_cols].copy()
    base_df = base_df.merge(rc_subset_df, on="query_name", how="left")
else:
    base_df["Rc"] = None
    base_df["Rc_selected_edges"] = None

base_df["Rc"] = base_df["Rc"].fillna(0).astype(int)
base_df["Rc_selected_edges"] = base_df["Rc_selected_edges"].apply(normalize_edge_list)

# ---------------------------------------------------------
# 6) Validações importantes
# ---------------------------------------------------------
missing_roots = base_df[base_df["selected_root"].isna()]["query_name"].tolist()
if missing_roots:
    raise ValueError(
        "Existem queries sem raiz definida em selected_roots_df: "
        f"{missing_roots}"
    )

invalid_root_rows = []
for _, row in base_df.iterrows():
    if row["selected_root"] not in row["entities"]:
        invalid_root_rows.append((row["query_name"], row["selected_root"], row["entities"]))

if invalid_root_rows:
    raise ValueError(
        "Existem queries com raiz fora do conjunto de entidades tocadas: "
        f"{invalid_root_rows}"
    )

# ---------------------------------------------------------
# 7) Calcular distâncias e profundidade por query
# ---------------------------------------------------------
depth_rows = []

for _, row in base_df.iterrows():
    query_name = row["query_name"]
    query_type = row["type"]
    generic_class = row["generic_class"]
    query_notes = row["notes"]

    entities_touched = row["entities"]
    selected_root = row["selected_root"]
    selected_edge_ids = row["Rc_selected_edges"]

    sub_adj = build_subgraph_adjacency_from_edge_ids(selected_edge_ids)

    entity_distances = {}
    unreachable_entities = []

    for entity in entities_touched:
        d = shortest_distance_in_subgraph(selected_root, entity, sub_adj)
        entity_distances[entity] = d
        if d is None:
            unreachable_entities.append(entity)

    valid_distances = [d for d in entity_distances.values() if d is not None]
    depth_value = max(valid_distances) if valid_distances else 0

    depth_rows.append({
        "query_name": query_name,
        "generic_class": generic_class,
        "query_type": query_type,
        "entities_touched": entities_touched,
        "selected_root": selected_root,
        "Rc": row["Rc"],
        "D_value": depth_value,
        "entity_distances_from_root": entity_distances,
        "unreachable_entities": unreachable_entities,
        "all_entities_reachable": len(unreachable_entities) == 0,
        "Rc_selected_edges": selected_edge_ids,
        "root_decision_rationale": row["root_decision_rationale"],
        "query_notes": query_notes
    })

depth_df = pd.DataFrame(depth_rows)

# ---------------------------------------------------------
# 8) Visões auxiliares
# ---------------------------------------------------------
depth_summary_df = depth_df[
    [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "Rc",
        "D_value",
        "all_entities_reachable",
        "unreachable_entities"
    ]
].copy()

print("Extração de D(E, er, Qi):")
display(depth_summary_df)

print("\nDetalhamento completo de profundidade:")
display(depth_df)

In [ ]:
# =========================================================
# BLOCK V6 — OBSERVED CARDINALITY (LIGHT VERSION)
# aligned to IMDb real via DuckDB
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = ["imdb_data_bundle"]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos do IMDb real."
        )

con = imdb_data_bundle.con

# ---------------------------------------------------------
# 2) Reduzir ruído no notebook
# ---------------------------------------------------------
try:
    con.execute("PRAGMA disable_progress_bar")
except:
    pass

# ---------------------------------------------------------
# 3) Helper function leve
# ---------------------------------------------------------
def summarize_observed_cardinality_sql(
    con,
    rel_name: str,
    rel_sql: str,
    source_col: str,
    target_col: str,
    source_entity: str,
    target_entity: str
) -> dict:
    """
    Resume cardinalidade observada de uma relação.
    Faz só o necessário para não sobrecarregar o notebook.
    """
    sql = f"""
    WITH rel AS (
        SELECT DISTINCT
            {source_col} AS source_id,
            {target_col} AS target_id
        FROM ({rel_sql}) x
        WHERE {source_col} IS NOT NULL
          AND {target_col} IS NOT NULL
    ),
    source_to_target AS (
        SELECT source_id, COUNT(*) AS n_targets
        FROM rel
        GROUP BY source_id
    ),
    target_to_source AS (
        SELECT target_id, COUNT(*) AS n_sources
        FROM rel
        GROUP BY target_id
    )
    SELECT
        '{rel_name}' AS relationship_name,
        '{source_entity}' AS source_entity,
        '{target_entity}' AS target_entity,
        (SELECT COUNT(*) FROM source_to_target) AS n_distinct_sources,
        (SELECT COUNT(*) FROM target_to_source) AS n_distinct_targets,
        ROUND((SELECT AVG(n_targets) FROM source_to_target), 4) AS avg_targets_per_source,
        (SELECT MAX(n_targets) FROM source_to_target) AS max_targets_per_source,
        ROUND((SELECT AVG(n_sources) FROM target_to_source), 4) AS avg_sources_per_target,
        (SELECT MAX(n_sources) FROM target_to_source) AS max_sources_per_target
    """
    row = con.execute(sql).df().iloc[0].to_dict()

    max_t = row["max_targets_per_source"]
    max_s = row["max_sources_per_target"]

    if max_t == 1 and max_s == 1:
        row["observed_pattern"] = "1:1"
    elif max_t > 1 and max_s == 1:
        row["observed_pattern"] = "1:N"
    elif max_t == 1 and max_s > 1:
        row["observed_pattern"] = "N:1"
    else:
        row["observed_pattern"] = "N:M"

    return row

# ---------------------------------------------------------
# 4) Só as arestas do cenário conceitual
# ---------------------------------------------------------
cardinality_rows = []

# has_genre : Genre -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="has_genre",
        rel_sql="""
            SELECT
                primary_genre,
                watchitem_id
            FROM imdb_watchitems
            WHERE primary_genre IS NOT NULL
        """,
        source_col="primary_genre",
        target_col="watchitem_id",
        source_entity="Genre",
        target_entity="WatchItem"
    )
)

# movie_is_watchitem : Movie -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="movie_is_watchitem",
        rel_sql="""
            SELECT
                watchitem_id AS movie_id,
                watchitem_id AS watchitem_id
            FROM imdb_movies
        """,
        source_col="movie_id",
        target_col="watchitem_id",
        source_entity="Movie",
        target_entity="WatchItem"
    )
)

# series_is_watchitem : Series -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="series_is_watchitem",
        rel_sql="""
            SELECT
                watchitem_id AS series_id,
                watchitem_id AS watchitem_id
            FROM imdb_series
        """,
        source_col="series_id",
        target_col="watchitem_id",
        source_entity="Series",
        target_entity="WatchItem"
    )
)

# episode_is_watchitem : Episode -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="episode_is_watchitem",
        rel_sql="""
            SELECT
                episode_watchitem_id AS episode_id,
                episode_watchitem_id AS watchitem_id
            FROM imdb_episodes
        """,
        source_col="episode_id",
        target_col="watchitem_id",
        source_entity="Episode",
        target_entity="WatchItem"
    )
)

# person_in_role : Person -> Role
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="person_in_role",
        rel_sql="""
            SELECT
                person_id,
                role_id
            FROM imdb_roles
        """,
        source_col="person_id",
        target_col="role_id",
        source_entity="Person",
        target_entity="Role"
    )
)

# role_of_watchitem : Role -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="role_of_watchitem",
        rel_sql="""
            SELECT
                role_id,
                watchitem_id
            FROM imdb_roles
        """,
        source_col="role_id",
        target_col="watchitem_id",
        source_entity="Role",
        target_entity="WatchItem"
    )
)

# series_contains_episode : Series -> Episode
cardinality_rows.append(
    summarize_observed_cardinality_sql(
        con=con,
        rel_name="series_contains_episode",
        rel_sql="""
            SELECT
                series_watchitem_id,
                episode_watchitem_id
            FROM imdb_episodes
            WHERE series_watchitem_id IS NOT NULL
        """,
        source_col="series_watchitem_id",
        target_col="episode_watchitem_id",
        source_entity="Series",
        target_entity="Episode"
    )
)

# ---------------------------------------------------------
# 5) DataFrame final
# ---------------------------------------------------------
cardinality_df = pd.DataFrame(cardinality_rows).sort_values(
    "relationship_name"
).reset_index(drop=True)

cardinality_map = {
    row["relationship_name"]: row
    for _, row in cardinality_df.iterrows()
}

print("Cardinalidade observada das arestas do cenário:")
display(cardinality_df)

In [ ]:
# =========================================================
# BLOCK V7 — SEMANTIC CLASSIFICATION OF RELATIONSHIPS
# aligned to cenário IMDb real QG1–QG10
# =========================================================

required_names = ["imdb_scenario"]
for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first a célula da INSTANCE 1."
        )

import pandas as pd

# ---------------------------------------------------------
# 1) Classificação semântica das arestas do cenário
#
# semantic_group:
#   - association
#   - associative
#   - containment
#
# semantic_detail:
#   - descriptive_association
#   - structural_association
#   - associative
#   - containment
#
# A ideia é manter compatibilidade com a matriz atual
# (que usa o núcleo association/associative/containment),
# mas sem perder o refinamento necessário para G2 e G3.
# ---------------------------------------------------------
relationship_semantics_rows = [
    {
        "relationship_name": "has_genre",
        "source": "Genre",
        "target": "WatchItem",
        "semantic_group": "association",
        "semantic_detail": "descriptive_association",
        "is_shared_descriptor": True,
        "is_structural_component": False,
        "is_associative_bridge": False,
        "is_containment": False,
        "notes": (
            "Genre funciona como descritor pequeno e compartilhado "
            "ligado à raiz WatchItem."
        ),
    },
    {
        "relationship_name": "movie_is_watchitem",
        "source": "Movie",
        "target": "WatchItem",
        "semantic_group": "association",
        "semantic_detail": "structural_association",
        "is_shared_descriptor": False,
        "is_structural_component": True,
        "is_associative_bridge": False,
        "is_containment": False,
        "notes": (
            "Movie é subtipo/componente estrutural próximo da raiz WatchItem."
        ),
    },
    {
        "relationship_name": "series_is_watchitem",
        "source": "Series",
        "target": "WatchItem",
        "semantic_group": "association",
        "semantic_detail": "structural_association",
        "is_shared_descriptor": False,
        "is_structural_component": True,
        "is_associative_bridge": False,
        "is_containment": False,
        "notes": (
            "Series é subtipo/componente estrutural próximo da raiz WatchItem."
        ),
    },
    {
        "relationship_name": "episode_is_watchitem",
        "source": "Episode",
        "target": "WatchItem",
        "semantic_group": "association",
        "semantic_detail": "structural_association",
        "is_shared_descriptor": False,
        "is_structural_component": True,
        "is_associative_bridge": False,
        "is_containment": False,
        "notes": (
            "Episode é subtipo/componente estrutural próximo da raiz WatchItem."
        ),
    },
    {
        "relationship_name": "person_in_role",
        "source": "Person",
        "target": "Role",
        "semantic_group": "associative",
        "semantic_detail": "associative",
        "is_shared_descriptor": False,
        "is_structural_component": False,
        "is_associative_bridge": True,
        "is_containment": False,
        "notes": (
            "Aresta do caminho associativo entre entidade final compartilhada "
            "e entidade associativa."
        ),
    },
    {
        "relationship_name": "role_of_watchitem",
        "source": "Role",
        "target": "WatchItem",
        "semantic_group": "associative",
        "semantic_detail": "associative",
        "is_shared_descriptor": False,
        "is_structural_component": False,
        "is_associative_bridge": True,
        "is_containment": False,
        "notes": (
            "Aresta do caminho associativo entre a entidade associativa "
            "e a raiz WatchItem."
        ),
    },
    {
        "relationship_name": "series_contains_episode",
        "source": "Series",
        "target": "Episode",
        "semantic_group": "containment",
        "semantic_detail": "containment",
        "is_shared_descriptor": False,
        "is_structural_component": False,
        "is_associative_bridge": False,
        "is_containment": True,
        "notes": (
            "Relação parte-todo explícita entre Series e Episode."
        ),
    },
]

relationship_semantics_df = pd.DataFrame(relationship_semantics_rows)

# ---------------------------------------------------------
# 2) Índices auxiliares para uso nos próximos blocos
# ---------------------------------------------------------
relationship_semantics_map = {
    row["relationship_name"]: row
    for row in relationship_semantics_rows
}

semantic_group_map = {
    row["relationship_name"]: row["semantic_group"]
    for row in relationship_semantics_rows
}

semantic_detail_map = {
    row["relationship_name"]: row["semantic_detail"]
    for row in relationship_semantics_rows
}

# ---------------------------------------------------------
# 3) Funções auxiliares
# ---------------------------------------------------------
def get_relationship_semantic_group(relationship_name: str) -> str:
    """
    Retorna o grupo semântico principal:
    association, associative ou containment.
    """
    if relationship_name not in semantic_group_map:
        raise KeyError(f"Relacionamento sem classificação semântica: {relationship_name}")
    return semantic_group_map[relationship_name]

def get_relationship_semantic_detail(relationship_name: str) -> str:
    """
    Retorna o detalhamento semântico fino:
    descriptive_association, structural_association,
    associative ou containment.
    """
    if relationship_name not in semantic_detail_map:
        raise KeyError(f"Relacionamento sem classificação semântica: {relationship_name}")
    return semantic_detail_map[relationship_name]

def get_relationship_semantic_record(relationship_name: str) -> dict:
    """
    Retorna o registro completo da classificação semântica da aresta.
    """
    if relationship_name not in relationship_semantics_map:
        raise KeyError(f"Relacionamento sem classificação semântica: {relationship_name}")
    return relationship_semantics_map[relationship_name]

# ---------------------------------------------------------
# 4) Validação simples: toda aresta do cenário deve estar classificada
# ---------------------------------------------------------
scenario_relationship_names = {
    rel["name"] for rel in imdb_scenario.relationships
}
classified_relationship_names = set(relationship_semantics_df["relationship_name"])

missing_in_semantics = sorted(scenario_relationship_names - classified_relationship_names)
extra_in_semantics = sorted(classified_relationship_names - scenario_relationship_names)

if missing_in_semantics:
    raise ValueError(
        "Existem relacionamentos no cenário sem classificação semântica: "
        f"{missing_in_semantics}"
    )

if extra_in_semantics:
    print("Warning: existem classificações semânticas sem uso no cenário atual:")
    print(extra_in_semantics)

# ---------------------------------------------------------
# 5) Visões auxiliares
# ---------------------------------------------------------
relationship_semantic_summary_df = relationship_semantics_df[
    [
        "relationship_name",
        "source",
        "target",
        "semantic_group",
        "semantic_detail",
        "is_shared_descriptor",
        "is_structural_component",
        "is_associative_bridge",
        "is_containment",
        "notes",
    ]
].copy()

print("Classificação semântica dos relacionamentos criada com sucesso.")
display(relationship_semantic_summary_df)

print("\nContagem por semantic_group:")
display(
    relationship_semantics_df.groupby("semantic_group").size().reset_index(name="n_relationships")
)

print("\nContagem por semantic_detail:")
display(
    relationship_semantics_df.groupby("semantic_detail").size().reset_index(name="n_relationships")
)

In [ ]:
# =========================================================
# BLOCK V8 — RELATIONSHIP TYPES TOUCHED BY EACH QUERY
# aligned to cenário IMDb real QG1–QG10
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "rc_df",
    "relationship_semantics_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Create edge_id compatível com o usado em Rc_selected_edges
# ---------------------------------------------------------
relationship_semantics_df = relationship_semantics_df.copy()
relationship_semantics_df["edge_id"] = relationship_semantics_df.apply(
    lambda row: f'{row["source"]} -- {row["target"]} [{row["relationship_name"]}]',
    axis=1
)

# ---------------------------------------------------------
# 3) Expandir os relacionamentos tocados por cada query
#    Agora guardamos:
#    - semantic_group   -> compatível com a matriz atual
#    - semantic_detail  -> refinamento útil para ativações futuras
# ---------------------------------------------------------
query_relationship_rows = []

for _, row in rc_df.iterrows():
    query_name = row["query_name"]
    selected_edges = row["Rc_selected_edges"]

    for edge_id in selected_edges:
        match = relationship_semantics_df[
            relationship_semantics_df["edge_id"] == edge_id
        ]

        if len(match) == 0:
            query_relationship_rows.append({
                "query_name": query_name,
                "edge_id": edge_id,
                "relationship_name": None,
                "semantic_group": None,
                "semantic_detail": None,
                "is_shared_descriptor": None,
                "is_structural_component": None,
                "is_associative_bridge": None,
                "is_containment": None,
                "notes": "Relacionamento não classificado"
            })
        else:
            for _, match_row in match.iterrows():
                query_relationship_rows.append({
                    "query_name": query_name,
                    "edge_id": edge_id,
                    "relationship_name": match_row["relationship_name"],
                    "semantic_group": match_row["semantic_group"],
                    "semantic_detail": match_row["semantic_detail"],
                    "is_shared_descriptor": bool(match_row["is_shared_descriptor"]),
                    "is_structural_component": bool(match_row["is_structural_component"]),
                    "is_associative_bridge": bool(match_row["is_associative_bridge"]),
                    "is_containment": bool(match_row["is_containment"]),
                    "notes": match_row["notes"]
                })

query_relationship_semantics_df = pd.DataFrame(query_relationship_rows)

print("Tipos semânticos dos relacionamentos tocados por cada query:")
display(query_relationship_semantics_df)

# ---------------------------------------------------------
# 4) Summary principal por query
#    Mantemos o nome semantic_type para compatibilidade com V11
# ---------------------------------------------------------
if not query_relationship_semantics_df.empty:
    semantic_summary_df = (
        query_relationship_semantics_df
        .groupby(["query_name", "semantic_group"], as_index=False)
        .size()
        .rename(columns={
            "semantic_group": "semantic_type",
            "size": "count"
        })
        .sort_values(["query_name", "semantic_type"])
        .reset_index(drop=True)
    )
else:
    semantic_summary_df = pd.DataFrame(
        columns=["query_name", "semantic_type", "count"]
    )

print("Summary semântico principal por query:")
display(semantic_summary_df)

# ---------------------------------------------------------
# 5) Summary detalhado por query
#    Este dataframe ainda não é exigido pelos blocos seguintes,
#    mas será útil para ativação metodológica mais refinada.
# ---------------------------------------------------------
if not query_relationship_semantics_df.empty:
    semantic_detail_summary_df = (
        query_relationship_semantics_df
        .groupby(["query_name", "semantic_detail"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["query_name", "semantic_detail"])
        .reset_index(drop=True)
    )
else:
    semantic_detail_summary_df = pd.DataFrame(
        columns=["query_name", "semantic_detail", "count"]
    )

print("Summary semântico detalhado por query:")
display(semantic_detail_summary_df)

# ---------------------------------------------------------
# 6) Flags estruturais auxiliares por query
#    Úteis para ativações futuras e para inspeção da matriz.
# ---------------------------------------------------------
if not query_relationship_semantics_df.empty:
    query_semantic_flags_df = (
        query_relationship_semantics_df
        .groupby("query_name", as_index=False)
        .agg(
            n_shared_descriptor_edges=("is_shared_descriptor", "sum"),
            n_structural_component_edges=("is_structural_component", "sum"),
            n_associative_bridge_edges=("is_associative_bridge", "sum"),
            n_containment_edges=("is_containment", "sum")
        )
    )
else:
    query_semantic_flags_df = pd.DataFrame(columns=[
        "query_name",
        "n_shared_descriptor_edges",
        "n_structural_component_edges",
        "n_associative_bridge_edges",
        "n_containment_edges"
    ])

print("Flags estruturais auxiliares por query:")
display(query_semantic_flags_df)

In [ ]:
# =========================================================
# BLOCK V9 — STRUCTURAL EDGE ANALYSIS BY DEPTH
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "depth_df",
    "edge_by_id"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Helper function:
# verifica se uma aresta pode ser absorvida pelo documento
# em uma profundidade de embutimento 'd'
# ---------------------------------------------------------
def edge_is_absorbed_at_depth(edge_id, entity_distances_from_root, d):
    """
    Uma aresta é considerada absorvida se as duas entidades
    de suas extremidades estiverem cobertas pela profundidade d.
    """
    edge = edge_by_id[edge_id]
    src = edge["source"]
    tgt = edge["target"]

    dist_src = entity_distances_from_root.get(src, None)
    dist_tgt = entity_distances_from_root.get(tgt, None)

    # Se alguma entidade não estiver no subgrafo da query,
    # tratamos como não absorvida.
    if dist_src is None or dist_tgt is None:
        return False

    return (dist_src <= d) and (dist_tgt <= d)

# ---------------------------------------------------------
# 3) Expandir aresta por aresta, profundidade por profundidade
# ---------------------------------------------------------
edge_depth_rows = []

for _, row in depth_df.iterrows():
    query_name = row["query_name"]
    selected_root = row["selected_root"]
    rc_value = int(row["Rc"])
    max_depth_for_query = int(row["D_value"])
    entity_distances = row["entity_distances_from_root"]
    selected_edge_ids = row["Rc_selected_edges"]

    # Para cada profundidade possível: 0 até D_value
    for d in range(0, max_depth_for_query + 1):
        for edge_id in selected_edge_ids:
            edge = edge_by_id[edge_id]

            absorbed = edge_is_absorbed_at_depth(
                edge_id=edge_id,
                entity_distances_from_root=entity_distances,
                d=d
            )

            edge_depth_rows.append({
                "query_name": query_name,
                "selected_root": selected_root,
                "test_depth": d,
                "Rc": rc_value,
                "edge_id": edge_id,
                "edge_source": edge["source"],
                "edge_target": edge["target"],
                "source_distance": entity_distances.get(edge["source"], None),
                "target_distance": entity_distances.get(edge["target"], None),
                "absorbed_by_document": absorbed
            })

edge_depth_df = pd.DataFrame(edge_depth_rows)

print("Análise estrutural das arestas por profundidade:")
display(edge_depth_df)

In [ ]:
# =========================================================
# BLOCK V10 — COMPUTE Re(Qi, er, D) AND DeltaR
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "depth_df",
    "edge_depth_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Casos com Rc > 0
# Aqui resumimos as queries que realmente têm arestas conceituais
# ---------------------------------------------------------
if not edge_depth_df.empty:
    re_summary_df = (
        edge_depth_df
        .groupby(["query_name", "selected_root", "test_depth", "Rc"], as_index=False)
        .agg(
            n_edges_total=("edge_id", "count"),
            n_edges_absorbed=("absorbed_by_document", "sum")
        )
    )

    # Re = arestas que continuam explícitas em execução
    re_summary_df["Re"] = re_summary_df["n_edges_total"] - re_summary_df["n_edges_absorbed"]

    # DeltaR = redução obtida pela estrutura documental
    re_summary_df["DeltaR"] = re_summary_df["Rc"] - re_summary_df["Re"]

else:
    re_summary_df = pd.DataFrame(columns=[
        "query_name", "selected_root", "test_depth", "Rc",
        "n_edges_total", "n_edges_absorbed", "Re", "DeltaR"
    ])

# ---------------------------------------------------------
# 3) Casos triviais com Rc = 0
# Para Q1 e Q2, não há relacionamentos para absorver
# ---------------------------------------------------------
zero_rc_rows = []

for _, row in depth_df.iterrows():
    if int(row["Rc"]) == 0:
        zero_rc_rows.append({
            "query_name": row["query_name"],
            "selected_root": row["selected_root"],
            "test_depth": 0,
            "Rc": 0,
            "n_edges_total": 0,
            "n_edges_absorbed": 0,
            "Re": 0,
            "DeltaR": 0
        })

zero_rc_df = pd.DataFrame(zero_rc_rows)

# ---------------------------------------------------------
# 4) Unir tudo
# ---------------------------------------------------------
re_full_df = pd.concat([re_summary_df, zero_rc_df], ignore_index=True)

re_full_df = re_full_df.sort_values(
    by=["query_name", "test_depth"]
).reset_index(drop=True)

print("Summary de Re(Qi, er, D) e DeltaR por profundidade:")
display(re_full_df)

In [ ]:
# =========================================================
# BLOCK V11 — BUILD COMPACT MATRIX PER QUERY
# aligned to workload QG1–QG10 e aos blocos V8, V9 e V10
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "depth_df",
    "semantic_summary_df",
    "semantic_detail_summary_df",
    "query_semantic_flags_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Encontrar automaticamente o dataframe do V10
#    (para não depender de um nome fixo)
# ---------------------------------------------------------
def find_reduction_summary_df():
    # nomes mais prováveis
    candidate_names = [
        "reduction_summary_df",
        "reduction_df",
        "delta_r_df",
        "deltaR_df",
        "re_df",
        "reduction_by_depth_df",
        "re_depth_summary_df"
    ]

    required_cols = {"query_name", "selected_root", "test_depth", "Rc", "Re", "DeltaR"}

    # primeiro tenta nomes conhecidos
    for name in candidate_names:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, pd.DataFrame) and required_cols.issubset(set(obj.columns)):
                return obj.copy(), name

    # depois procura em todos os dataframes do notebook
    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            cols = set(obj.columns)
            if required_cols.issubset(cols):
                return obj.copy(), name

    raise NameError(
        "Não encontrei automaticamente o dataframe do V10. "
        "Ele precisa ter pelo menos as colunas: "
        "{'query_name', 'selected_root', 'test_depth', 'Rc', 'Re', 'DeltaR'}."
    )

reduction_summary_df, reduction_source_name = find_reduction_summary_df()

print(f"DataFrame do V10 detectado automaticamente: {reduction_source_name}")

# ---------------------------------------------------------
# 3) Preparar DeltaRratio
# ---------------------------------------------------------
reduction_summary_df = reduction_summary_df.copy()

if "DeltaRratio" not in reduction_summary_df.columns:
    reduction_summary_df["DeltaRratio"] = reduction_summary_df.apply(
        lambda row: (row["DeltaR"] / row["Rc"]) if row["Rc"] > 0 else pd.NA,
        axis=1
    )

# ---------------------------------------------------------
# 4) Escolher a profundidade documental compacta por query
#
# Regra metodológica:
# - menor profundidade testada com Re = 0
# - se nenhuma tiver Re = 0, manter a maior profundidade testada
# ---------------------------------------------------------
selected_depth_rows = []

for query_name, group in reduction_summary_df.groupby("query_name"):
    group_sorted = group.sort_values("test_depth").reset_index(drop=True)

    zero_re_group = group_sorted[group_sorted["Re"] == 0]

    if len(zero_re_group) > 0:
        selected_row = zero_re_group.iloc[0].copy()
        depth_selection_reason = "min_depth_with_total_absorption"
    else:
        selected_row = group_sorted.iloc[-1].copy()
        depth_selection_reason = "max_tested_depth_no_total_absorption"

    selected_depth_rows.append({
        "query_name": selected_row["query_name"],
        "selected_root_from_reduction": selected_row["selected_root"],
        "Rc_from_reduction": int(selected_row["Rc"]),
        "document_depth_selected": int(selected_row["test_depth"]),
        "Re_selected": int(selected_row["Re"]),
        "DeltaR_selected": int(selected_row["DeltaR"]),
        "DeltaRratio_selected": selected_row["DeltaRratio"],
        "depth_selection_reason": depth_selection_reason
    })

selected_depth_df = pd.DataFrame(selected_depth_rows)

# ---------------------------------------------------------
# 5) Pivot do resumo semântico principal
#    semantic_type: association / associative / containment
# ---------------------------------------------------------
if semantic_summary_df.empty:
    semantic_group_pivot_df = pd.DataFrame(columns=[
        "query_name",
        "n_association",
        "n_associative",
        "n_containment"
    ])
else:
    semantic_group_pivot_df = (
        semantic_summary_df
        .pivot_table(
            index="query_name",
            columns="semantic_type",
            values="count",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in ["association", "associative", "containment"]:
        if col not in semantic_group_pivot_df.columns:
            semantic_group_pivot_df[col] = 0

    semantic_group_pivot_df = semantic_group_pivot_df.rename(columns={
        "association": "n_association",
        "associative": "n_associative",
        "containment": "n_containment"
    })

# ---------------------------------------------------------
# 6) Pivot do resumo semântico detalhado
# ---------------------------------------------------------
if semantic_detail_summary_df.empty:
    semantic_detail_pivot_df = pd.DataFrame(columns=[
        "query_name",
        "n_descriptive_association",
        "n_structural_association",
        "n_associative_detail",
        "n_containment_detail"
    ])
else:
    semantic_detail_pivot_df = (
        semantic_detail_summary_df
        .pivot_table(
            index="query_name",
            columns="semantic_detail",
            values="count",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in [
        "descriptive_association",
        "structural_association",
        "associative",
        "containment"
    ]:
        if col not in semantic_detail_pivot_df.columns:
            semantic_detail_pivot_df[col] = 0

    semantic_detail_pivot_df = semantic_detail_pivot_df.rename(columns={
        "descriptive_association": "n_descriptive_association",
        "structural_association": "n_structural_association",
        "associative": "n_associative_detail",
        "containment": "n_containment_detail"
    })

# ---------------------------------------------------------
# 7) Base da matriz = depth_df
# ---------------------------------------------------------
matrix_compact_df = depth_df[
    [
        "query_name",
        "generic_class",
        "query_type",
        "entities_touched",
        "selected_root",
        "Rc",
        "D_value",
        "entity_distances_from_root",
        "all_entities_reachable",
        "Rc_selected_edges",
        "root_decision_rationale",
        "query_notes"
    ]
].copy()

# ---------------------------------------------------------
# 8) Merge com profundidade documental selecionada (V10)
# ---------------------------------------------------------
matrix_compact_df = matrix_compact_df.merge(
    selected_depth_df,
    on="query_name",
    how="left"
)

# checagem de consistência entre V5 e V10
matrix_compact_df["root_consistent_v5_v10"] = (
    matrix_compact_df["selected_root"] == matrix_compact_df["selected_root_from_reduction"]
)

matrix_compact_df["Rc_consistent_v5_v10"] = (
    matrix_compact_df["Rc"] == matrix_compact_df["Rc_from_reduction"]
)

# ---------------------------------------------------------
# 9) Merge com contagens semânticas
# ---------------------------------------------------------
matrix_compact_df = matrix_compact_df.merge(
    semantic_group_pivot_df,
    on="query_name",
    how="left"
)

matrix_compact_df = matrix_compact_df.merge(
    semantic_detail_pivot_df,
    on="query_name",
    how="left"
)

matrix_compact_df = matrix_compact_df.merge(
    query_semantic_flags_df,
    on="query_name",
    how="left"
)

# ---------------------------------------------------------
# 10) Preencher ausências com zero
# ---------------------------------------------------------
numeric_fill_zero_cols = [
    "n_association",
    "n_associative",
    "n_containment",
    "n_descriptive_association",
    "n_structural_association",
    "n_associative_detail",
    "n_containment_detail",
    "n_shared_descriptor_edges",
    "n_structural_component_edges",
    "n_associative_bridge_edges",
    "n_containment_edges"
]

for col in numeric_fill_zero_cols:
    if col not in matrix_compact_df.columns:
        matrix_compact_df[col] = 0

matrix_compact_df[numeric_fill_zero_cols] = (
    matrix_compact_df[numeric_fill_zero_cols]
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 11) Tipo semântico dominante (grupo principal)
# ---------------------------------------------------------
def get_dominant_semantic_group(row):
    candidates = {
        "association": row["n_association"],
        "associative": row["n_associative"],
        "containment": row["n_containment"]
    }
    max_value = max(candidates.values())
    if max_value == 0:
        return "none"

    winners = [k for k, v in candidates.items() if v == max_value]
    if len(winners) == 1:
        return winners[0]
    return "mixed_" + "_".join(sorted(winners))

matrix_compact_df["dominant_semantic_group"] = matrix_compact_df.apply(
    get_dominant_semantic_group,
    axis=1
)

# ---------------------------------------------------------
# 12) Tipo semântico dominante (detalhe fino)
# ---------------------------------------------------------
def get_dominant_semantic_detail(row):
    candidates = {
        "descriptive_association": row["n_descriptive_association"],
        "structural_association": row["n_structural_association"],
        "associative": row["n_associative_detail"],
        "containment": row["n_containment_detail"]
    }
    max_value = max(candidates.values())
    if max_value == 0:
        return "none"

    winners = [k for k, v in candidates.items() if v == max_value]
    if len(winners) == 1:
        return winners[0]
    return "mixed_" + "_".join(sorted(winners))

matrix_compact_df["dominant_semantic_detail"] = matrix_compact_df.apply(
    get_dominant_semantic_detail,
    axis=1
)

# ---------------------------------------------------------
# 13) Flags lógicas úteis
# ---------------------------------------------------------
matrix_compact_df["has_association"] = matrix_compact_df["n_association"] > 0
matrix_compact_df["has_associative"] = matrix_compact_df["n_associative"] > 0
matrix_compact_df["has_containment"] = matrix_compact_df["n_containment"] > 0

matrix_compact_df["has_shared_descriptor"] = (
    matrix_compact_df["n_shared_descriptor_edges"] > 0
)
matrix_compact_df["has_structural_component"] = (
    matrix_compact_df["n_structural_component_edges"] > 0
)
matrix_compact_df["has_associative_bridge"] = (
    matrix_compact_df["n_associative_bridge_edges"] > 0
)

# ---------------------------------------------------------
# 14) Normalizar DeltaRratio_selected
# ---------------------------------------------------------
matrix_compact_df["DeltaRratio_selected"] = matrix_compact_df["DeltaRratio_selected"].astype("object")

# ---------------------------------------------------------
# 15) Ordenação final
# ---------------------------------------------------------
matrix_compact_df = matrix_compact_df.sort_values(
    ["generic_class", "query_name"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 16) Summary enxuto para inspeção
# ---------------------------------------------------------
matrix_compact_summary_df = matrix_compact_df[
    [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "Rc",
        "D_value",
        "document_depth_selected",
        "Re_selected",
        "DeltaR_selected",
        "DeltaRratio_selected",
        "dominant_semantic_group",
        "dominant_semantic_detail",
        "n_association",
        "n_associative",
        "n_containment",
        "n_shared_descriptor_edges",
        "n_structural_component_edges",
        "n_associative_bridge_edges",
        "n_containment_edges",
        "depth_selection_reason",
        "root_consistent_v5_v10",
        "Rc_consistent_v5_v10"
    ]
].copy()

print("Matriz compacta parcial por query:")
display(matrix_compact_summary_df)

print("\nMatriz compacta detalhada:")
display(matrix_compact_df)

In [ ]:
# =========================================================
# BLOCK V12 — IMDb ANALYTICAL MATRIX (EXPANDED BY DEPTH)
# aligned to workload QG1–QG10 e aos blocos V8, V9, V10 e V11
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "imdb_scenario",
    "depth_df",
    "semantic_summary_df",
    "semantic_detail_summary_df",
    "query_semantic_flags_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Detectar automaticamente o dataframe do V10
# ---------------------------------------------------------
def find_reduction_summary_df():
    candidate_names = [
        "reduction_summary_df",
        "reduction_df",
        "delta_r_df",
        "deltaR_df",
        "re_df",
        "reduction_by_depth_df",
        "re_depth_summary_df"
    ]

    required_cols = {"query_name", "selected_root", "test_depth", "Rc", "Re", "DeltaR"}

    for name in candidate_names:
        if name in globals():
            obj = globals()[name]
            if isinstance(obj, pd.DataFrame) and required_cols.issubset(set(obj.columns)):
                return obj.copy(), name

    for name, obj in globals().items():
        if isinstance(obj, pd.DataFrame):
            cols = set(obj.columns)
            if required_cols.issubset(cols):
                return obj.copy(), name

    raise NameError(
        "Não encontrei automaticamente o dataframe do V10. "
        "Ele precisa ter pelo menos as colunas: "
        "{'query_name', 'selected_root', 'test_depth', 'Rc', 'Re', 'DeltaR'}."
    )

reduction_summary_df, reduction_source_name = find_reduction_summary_df()

print(f"DataFrame do V10 detectado automaticamente: {reduction_source_name}")

# ---------------------------------------------------------
# 3) Current workload base
# ---------------------------------------------------------
workload_base_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

if "query_name" not in workload_base_df.columns:
    if "name" in workload_base_df.columns:
        workload_base_df = workload_base_df.rename(columns={"name": "query_name"})
    else:
        raise ValueError("O workload atual não possui 'query_name' nem 'name'.")

if "entities" not in workload_base_df.columns:
    raise ValueError("O workload atual precisa ter a coluna 'entities'.")

if "type" not in workload_base_df.columns:
    workload_base_df["type"] = None

if "generic_class" not in workload_base_df.columns:
    workload_base_df["generic_class"] = None

if "notes" not in workload_base_df.columns:
    workload_base_df["notes"] = None

workload_base_df["Qi_entities_touched"] = workload_base_df["entities"]
workload_base_df["Qi_n_entities_touched"] = workload_base_df["Qi_entities_touched"].apply(len)
workload_base_df = workload_base_df.rename(columns={
    "type": "query_type",
    "notes": "query_notes"
})

workload_base_df = workload_base_df[
    [
        "query_name",
        "generic_class",
        "query_type",
        "Qi_entities_touched",
        "Qi_n_entities_touched",
        "abstract_query",
        "query_notes"
    ]
].copy()

# ---------------------------------------------------------
# 4) Pivot do resumo semântico principal
# ---------------------------------------------------------
if semantic_summary_df.empty:
    semantic_group_pivot_df = pd.DataFrame(columns=[
        "query_name",
        "n_association",
        "n_associative",
        "n_containment"
    ])
else:
    semantic_group_pivot_df = (
        semantic_summary_df
        .pivot_table(
            index="query_name",
            columns="semantic_type",
            values="count",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in ["association", "associative", "containment"]:
        if col not in semantic_group_pivot_df.columns:
            semantic_group_pivot_df[col] = 0

    semantic_group_pivot_df = semantic_group_pivot_df.rename(columns={
        "association": "n_association",
        "associative": "n_associative",
        "containment": "n_containment"
    })

# ---------------------------------------------------------
# 5) Pivot do resumo semântico detalhado
# ---------------------------------------------------------
if semantic_detail_summary_df.empty:
    semantic_detail_pivot_df = pd.DataFrame(columns=[
        "query_name",
        "n_descriptive_association",
        "n_structural_association",
        "n_associative_detail",
        "n_containment_detail"
    ])
else:
    semantic_detail_pivot_df = (
        semantic_detail_summary_df
        .pivot_table(
            index="query_name",
            columns="semantic_detail",
            values="count",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for col in [
        "descriptive_association",
        "structural_association",
        "associative",
        "containment"
    ]:
        if col not in semantic_detail_pivot_df.columns:
            semantic_detail_pivot_df[col] = 0

    semantic_detail_pivot_df = semantic_detail_pivot_df.rename(columns={
        "descriptive_association": "n_descriptive_association",
        "structural_association": "n_structural_association",
        "associative": "n_associative_detail",
        "containment": "n_containment_detail"
    })

# ---------------------------------------------------------
# 6) Estrutura por query (independente da profundidade)
# ---------------------------------------------------------
query_structure_df = depth_df[
    [
        "query_name",
        "selected_root",
        "Rc",
        "D_value",
        "entity_distances_from_root",
        "all_entities_reachable",
        "Rc_selected_edges",
        "root_decision_rationale"
    ]
].copy()

query_structure_df = query_structure_df.merge(
    semantic_group_pivot_df,
    on="query_name",
    how="left"
)

query_structure_df = query_structure_df.merge(
    semantic_detail_pivot_df,
    on="query_name",
    how="left"
)

query_structure_df = query_structure_df.merge(
    query_semantic_flags_df,
    on="query_name",
    how="left"
)

# preencher ausências semânticas com zero
semantic_numeric_cols = [
    "n_association",
    "n_associative",
    "n_containment",
    "n_descriptive_association",
    "n_structural_association",
    "n_associative_detail",
    "n_containment_detail",
    "n_shared_descriptor_edges",
    "n_structural_component_edges",
    "n_associative_bridge_edges",
    "n_containment_edges"
]

for col in semantic_numeric_cols:
    if col not in query_structure_df.columns:
        query_structure_df[col] = 0

query_structure_df[semantic_numeric_cols] = (
    query_structure_df[semantic_numeric_cols]
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 7) Base expandida por profundidade
# ---------------------------------------------------------
analytical_matrix_df = (
    workload_base_df
    .merge(query_structure_df, on="query_name", how="left")
    .merge(
        reduction_summary_df[
            ["query_name", "selected_root", "test_depth", "Rc", "Re", "DeltaR"]
        ].rename(columns={
            "selected_root": "selected_root_from_reduction",
            "Rc": "Rc_from_reduction"
        }),
        on="query_name",
        how="left"
    )
)

# ---------------------------------------------------------
# 8) DeltaRratio
# ---------------------------------------------------------
def safe_ratio(delta_r, rc):
    if pd.isna(rc) or rc == 0:
        return np.nan
    return delta_r / rc

analytical_matrix_df["DeltaR_ratio"] = analytical_matrix_df.apply(
    lambda row: safe_ratio(row["DeltaR"], row["Rc_from_reduction"]),
    axis=1
)

# ---------------------------------------------------------
# 9) Rótulo da cobertura estrutural
# ---------------------------------------------------------
def classify_structural_coverage(row):
    rc = row["Rc_from_reduction"]
    re = row["Re"]

    if pd.isna(rc) or rc == 0:
        return "No structural traversal"
    if re == 0:
        return "Full absorption"
    if re < rc:
        return "Partial absorption"
    return "No absorption"

analytical_matrix_df["document_structural_coverage"] = analytical_matrix_df.apply(
    classify_structural_coverage,
    axis=1
)

# ---------------------------------------------------------
# 10) Tipo semântico dominante
# ---------------------------------------------------------
def get_dominant_semantic_group(row):
    candidates = {
        "association": row["n_association"],
        "associative": row["n_associative"],
        "containment": row["n_containment"]
    }
    max_value = max(candidates.values())
    if max_value == 0:
        return "none"

    winners = [k for k, v in candidates.items() if v == max_value]
    if len(winners) == 1:
        return winners[0]
    return "mixed_" + "_".join(sorted(winners))

analytical_matrix_df["dominant_semantic_group"] = analytical_matrix_df.apply(
    get_dominant_semantic_group,
    axis=1
)

def get_dominant_semantic_detail(row):
    candidates = {
        "descriptive_association": row["n_descriptive_association"],
        "structural_association": row["n_structural_association"],
        "associative": row["n_associative_detail"],
        "containment": row["n_containment_detail"]
    }
    max_value = max(candidates.values())
    if max_value == 0:
        return "none"

    winners = [k for k, v in candidates.items() if v == max_value]
    if len(winners) == 1:
        return winners[0]
    return "mixed_" + "_".join(sorted(winners))

analytical_matrix_df["dominant_semantic_detail"] = analytical_matrix_df.apply(
    get_dominant_semantic_detail,
    axis=1
)

# ---------------------------------------------------------
# 11) Checksgens de consistência entre V5 e V10
# ---------------------------------------------------------
analytical_matrix_df["root_consistent_v5_v10"] = (
    analytical_matrix_df["selected_root"] == analytical_matrix_df["selected_root_from_reduction"]
)

analytical_matrix_df["Rc_consistent_v5_v10"] = (
    analytical_matrix_df["Rc"] == analytical_matrix_df["Rc_from_reduction"]
)

# ---------------------------------------------------------
# 12) Sort para leitura
# ---------------------------------------------------------
analytical_matrix_df = analytical_matrix_df.sort_values(
    by=["query_name", "test_depth"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 13) Visão resumida
# ---------------------------------------------------------
analytical_matrix_summary_df = analytical_matrix_df[
    [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "test_depth",
        "Rc",
        "D_value",
        "Re",
        "DeltaR",
        "DeltaR_ratio",
        "document_structural_coverage",
        "dominant_semantic_group",
        "dominant_semantic_detail",
        "n_shared_descriptor_edges",
        "n_structural_component_edges",
        "n_associative_bridge_edges",
        "n_containment_edges",
        "root_consistent_v5_v10",
        "Rc_consistent_v5_v10"
    ]
].copy()

print("Consolidated analytical matrix for IMDb (expandida por profundidade):")
display(analytical_matrix_summary_df)

print("\nMatriz analítica expandida detalhada:")
display(analytical_matrix_df)

In [ ]:
# =========================================================
# BLOCK V13 — COMPACT ANALYTICAL MATRIX PER QUERY
# derivada do V12 + checagem contra o V11
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função para escolher a linha representativa da query
#
# Regra metodológica:
# - se houver profundidade com Re = 0, pegar a menor delas
# - senão, pegar a maior profundidade testada
# ---------------------------------------------------------
def select_representative_row(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("test_depth").reset_index(drop=True)

    full_absorption = group[group["Re"] == 0]

    if not full_absorption.empty:
        selected = full_absorption.iloc[0].copy()
        selected["depth_selection_reason_v13"] = "min_depth_with_total_absorption"
    else:
        selected = group.iloc[-1].copy()
        selected["depth_selection_reason_v13"] = "max_tested_depth_no_total_absorption"

    return selected

# ---------------------------------------------------------
# 3) Apply por query
# ---------------------------------------------------------
compact_rows = []

for query_name, group in analytical_matrix_df.groupby("query_name"):
    selected = select_representative_row(group)
    compact_rows.append(selected)

analytical_matrix_compact_df = pd.DataFrame(compact_rows).reset_index(drop=True)

# ---------------------------------------------------------
# 4) Renomear colunas para deixar a leitura explícita
# ---------------------------------------------------------
analytical_matrix_compact_df = analytical_matrix_compact_df.rename(columns={
    "test_depth": "selected_document_depth",
    "Re": "selected_Re",
    "DeltaR": "selected_DeltaR",
    "DeltaR_ratio": "selected_DeltaR_ratio",
    "document_structural_coverage": "selected_structural_coverage"
})

# ---------------------------------------------------------
# 5) Sort
# ---------------------------------------------------------
sort_cols = [c for c in ["generic_class", "query_name"] if c in analytical_matrix_compact_df.columns]
if sort_cols:
    analytical_matrix_compact_df = analytical_matrix_compact_df.sort_values(
        by=sort_cols
    ).reset_index(drop=True)

# ---------------------------------------------------------
# 6) Summary enxuto
# ---------------------------------------------------------
compact_summary_cols = [
    c for c in [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "Rc",
        "D_value",
        "selected_document_depth",
        "selected_Re",
        "selected_DeltaR",
        "selected_DeltaR_ratio",
        "selected_structural_coverage",
        "dominant_semantic_group",
        "dominant_semantic_detail",
        "n_shared_descriptor_edges",
        "n_structural_component_edges",
        "n_associative_bridge_edges",
        "n_containment_edges",
        "depth_selection_reason_v13"
    ]
    if c in analytical_matrix_compact_df.columns
]

analytical_matrix_compact_summary_df = analytical_matrix_compact_df[
    compact_summary_cols
].copy()

print("Compact analytical matrix for IMDb (derivada do V12):")
display(analytical_matrix_compact_summary_df)

print("\nMatriz analítica compacta detalhada:")
display(analytical_matrix_compact_df)

# ---------------------------------------------------------
# 7) Checksgem opcional de consistência com o V11
# ---------------------------------------------------------
if "matrix_compact_df" in globals():
    compare_left = analytical_matrix_compact_df.copy()
    compare_right = matrix_compact_df.copy()

    # renomear colunas do V11 para evitar colisão
    rename_map_right = {}

    if "selected_root" in compare_right.columns:
        rename_map_right["selected_root"] = "selected_root_v11"
    if "Rc" in compare_right.columns:
        rename_map_right["Rc"] = "Rc_v11"
    if "D_value" in compare_right.columns:
        rename_map_right["D_value"] = "D_value_v11"
    if "document_depth_selected" in compare_right.columns:
        rename_map_right["document_depth_selected"] = "selected_document_depth_v11"
    if "Re_selected" in compare_right.columns:
        rename_map_right["Re_selected"] = "selected_Re_v11"
    if "DeltaR_selected" in compare_right.columns:
        rename_map_right["DeltaR_selected"] = "selected_DeltaR_v11"
    if "DeltaRratio_selected" in compare_right.columns:
        rename_map_right["DeltaRratio_selected"] = "selected_DeltaR_ratio_v11"
    if "depth_selection_reason" in compare_right.columns:
        rename_map_right["depth_selection_reason"] = "depth_selection_reason_v11"

    compare_right = compare_right.rename(columns=rename_map_right)

    compare_cols_right = [
        c for c in [
            "query_name",
            "selected_root_v11",
            "Rc_v11",
            "D_value_v11",
            "selected_document_depth_v11",
            "selected_Re_v11",
            "selected_DeltaR_v11",
            "selected_DeltaR_ratio_v11",
            "depth_selection_reason_v11"
        ]
        if c in compare_right.columns
    ]

    compare_df = compare_left.merge(
        compare_right[compare_cols_right],
        on="query_name",
        how="left"
    )

    # comparações seguras
    if "selected_root" in compare_df.columns and "selected_root_v11" in compare_df.columns:
        compare_df["selected_root_match_v11"] = (
            compare_df["selected_root"] == compare_df["selected_root_v11"]
        )
    else:
        compare_df["selected_root_match_v11"] = pd.NA

    if "Rc" in compare_df.columns and "Rc_v11" in compare_df.columns:
        compare_df["Rc_match_v11"] = (
            compare_df["Rc"] == compare_df["Rc_v11"]
        )
    else:
        compare_df["Rc_match_v11"] = pd.NA

    if "D_value" in compare_df.columns and "D_value_v11" in compare_df.columns:
        compare_df["D_value_match_v11"] = (
            compare_df["D_value"] == compare_df["D_value_v11"]
        )
    else:
        compare_df["D_value_match_v11"] = pd.NA

    if "selected_document_depth" in compare_df.columns and "selected_document_depth_v11" in compare_df.columns:
        compare_df["selected_document_depth_match_v11"] = (
            compare_df["selected_document_depth"] == compare_df["selected_document_depth_v11"]
        )
    else:
        compare_df["selected_document_depth_match_v11"] = pd.NA

    if "selected_Re" in compare_df.columns and "selected_Re_v11" in compare_df.columns:
        compare_df["selected_Re_match_v11"] = (
            compare_df["selected_Re"] == compare_df["selected_Re_v11"]
        )
    else:
        compare_df["selected_Re_match_v11"] = pd.NA

    if "selected_DeltaR" in compare_df.columns and "selected_DeltaR_v11" in compare_df.columns:
        compare_df["selected_DeltaR_match_v11"] = (
            compare_df["selected_DeltaR"] == compare_df["selected_DeltaR_v11"]
        )
    else:
        compare_df["selected_DeltaR_match_v11"] = pd.NA

    compare_summary_cols = [
        c for c in [
            "query_name",
            "selected_root",
            "selected_root_v11",
            "selected_root_match_v11",
            "Rc",
            "Rc_v11",
            "Rc_match_v11",
            "D_value",
            "D_value_v11",
            "D_value_match_v11",
            "selected_document_depth",
            "selected_document_depth_v11",
            "selected_document_depth_match_v11",
            "selected_Re",
            "selected_Re_v11",
            "selected_Re_match_v11",
            "selected_DeltaR",
            "selected_DeltaR_v11",
            "selected_DeltaR_match_v11",
            "depth_selection_reason_v13",
            "depth_selection_reason_v11"
        ]
        if c in compare_df.columns
    ]

    analytical_matrix_compact_consistency_df = compare_df[compare_summary_cols].copy()

    print("\nChecksgem de consistência entre V13 (derivado do V12) e V11:")
    display(analytical_matrix_compact_consistency_df)
else:
    print("\nWarning: 'matrix_compact_df' não existe no ambiente; checagem com V11 foi ignorada.")

In [ ]:
# =========================================================
# BLOCK V14 — ENTITIES EXPLICITLY MODIFIED BY QUERY
# aligned to workload QG1–QG10
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
required_names = ["imdb_scenario"]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first a célula da INSTANCE 1."
        )

# ---------------------------------------------------------
# 2) Current workload base
# ---------------------------------------------------------
workload_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

if "query_name" not in workload_df.columns:
    if "name" in workload_df.columns:
        workload_df = workload_df.rename(columns={"name": "query_name"})
    else:
        raise ValueError("O workload atual não possui 'query_name' nem 'name'.")

if "entities" not in workload_df.columns:
    raise ValueError("O workload atual precisa ter a coluna 'entities'.")

if "type" not in workload_df.columns:
    workload_df["type"] = None

if "generic_class" not in workload_df.columns:
    workload_df["generic_class"] = None

if "notes" not in workload_df.columns:
    workload_df["notes"] = None

# ---------------------------------------------------------
# 3) Definição manual-assistida das entidades modificadas
#
# Adopted rule:
# - only entities explicitly inserted, updated, or removed are included
# - entities only read/referenced are excluded
# ---------------------------------------------------------
query_update_map_rows = [
    {
        "query_name": "QG1_WatchItemById",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG2_WatchItemByTitle",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG3_RecommendationByGenreAndSubtype",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG4_AllPersonsOfTypeForWatchItem",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG5_AllPersonsForEpisodesOfSeries",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG6_EpisodesOfSeries",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG7_UpdateWatchItemMetadata",
        "updated_entities": ["WatchItem"],
        "update_rationale": (
            "QG7 representa escrita isolada. A query atualiza explicitamente "
            "atributos de WatchItem sem criar novo relacionamento."
        )
    },
    {
        "query_name": "QG8_AddPersonRoleToWatchItem",
        "updated_entities": ["Person", "Role"],
        "update_rationale": (
            "QG8 representa escrita com criação de relacionamento. "
            "A query cria/insere Person quando necessário e cria Role para "
            "ligar Person ao WatchItem. WatchItem é referenciado, mas não "
            "é atualizado explicitamente."
        )
    },
    {
        "query_name": "QG9_TopRatedSeriesByGenre",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
    {
        "query_name": "QG10_AdvancedSearchWatchItems",
        "updated_entities": [],
        "update_rationale": "Read-only query."
    },
]

query_update_map_df = pd.DataFrame(query_update_map_rows)

# ---------------------------------------------------------
# 4) Validar cobertura do workload
# ---------------------------------------------------------
workload_query_names = set(workload_df["query_name"])
map_query_names = set(query_update_map_df["query_name"])

missing_in_update_map = sorted(workload_query_names - map_query_names)
extra_in_update_map = sorted(map_query_names - workload_query_names)

if missing_in_update_map:
    raise ValueError(
        "Existem queries do workload sem definição de entidades modificadas: "
        f"{missing_in_update_map}"
    )

if extra_in_update_map:
    print("Warning: existem queries no mapa de updates que não estão no workload atual:")
    print(extra_in_update_map)

# ---------------------------------------------------------
# 5) Mesclar com o workload para inspeção
# ---------------------------------------------------------
query_update_map_df = workload_df[
    ["query_name", "generic_class", "type", "entities", "notes"]
].merge(
    query_update_map_df,
    on="query_name",
    how="left"
).rename(columns={
    "type": "query_type",
    "entities": "entities_touched",
    "notes": "query_notes"
})

query_update_map_df["n_updated_entities"] = query_update_map_df["updated_entities"].apply(len)
query_update_map_df["is_write_query"] = query_update_map_df["n_updated_entities"] > 0

print("Entidades explicitamente modificadas por query:")
display(query_update_map_df)

In [ ]:
# =========================================================
# BLOCK V15 — COMPUTE UPDATE VOLATILITY PER ENTITY
# aligned to workload QG1–QG10
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "imdb_scenario",
    "query_update_map_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Current workload base
# ---------------------------------------------------------
workload_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

if "query_name" not in workload_df.columns:
    if "name" in workload_df.columns:
        workload_df = workload_df.rename(columns={"name": "query_name"})
    else:
        raise ValueError("O workload atual não possui 'query_name' nem 'name'.")

# ---------------------------------------------------------
# 3) Define pesos/frequências do workload
# Neste experimento: peso igual para todas as queries
# ---------------------------------------------------------
query_frequency_df = workload_df[["query_name"]].copy()
query_frequency_df["query_weight"] = 1.0

# ---------------------------------------------------------
# 4) Explodir a relação query -> entidades modificadas
# W(e, Qi) = 1 se Qi modifica explicitamente e; 0 caso contrário
# ---------------------------------------------------------
all_entities = list(imdb_scenario.entities)

expanded_rows = []

for _, row in query_update_map_df.iterrows():
    query_name = row["query_name"]
    updated_entities = row["updated_entities"]

    for entity in all_entities:
        expanded_rows.append({
            "query_name": query_name,
            "entity": entity,
            "writes_entity": 1 if entity in updated_entities else 0
        })

query_entity_write_df = pd.DataFrame(expanded_rows)

# junta com os pesos
query_entity_write_df = query_entity_write_df.merge(
    query_frequency_df,
    on="query_name",
    how="left"
)

if query_entity_write_df["query_weight"].isna().any():
    missing_weight_queries = sorted(
        query_entity_write_df.loc[
            query_entity_write_df["query_weight"].isna(), "query_name"
        ].unique().tolist()
    )
    raise ValueError(
        "Existem queries sem peso definido em query_frequency_df: "
        f"{missing_weight_queries}"
    )

print("Matriz query-entidade de escrita:")
display(query_entity_write_df.head(30))

# ---------------------------------------------------------
# 5) Calcular upd(e)
# upd(e) = sum_i f_i * W(e, Qi) / sum_i f_i
# ---------------------------------------------------------
total_weight = query_frequency_df["query_weight"].sum()

entity_update_volatility_df = (
    query_entity_write_df
    .assign(weighted_write=lambda df: df["writes_entity"] * df["query_weight"])
    .groupby("entity", as_index=False)
    .agg(
        total_weighted_writes=("weighted_write", "sum")
    )
)

entity_update_volatility_df["update_volatility_score"] = (
    entity_update_volatility_df["total_weighted_writes"] / total_weight
)

entity_update_volatility_df = entity_update_volatility_df.sort_values(
    by=["update_volatility_score", "entity"],
    ascending=[False, True]
).reset_index(drop=True)

print("Update volatility por entidade:")
display(entity_update_volatility_df)

# ---------------------------------------------------------
# 6) Visão auxiliar interpretável
# ---------------------------------------------------------
entity_update_volatility_df["update_volatility_label"] = entity_update_volatility_df[
    "update_volatility_score"
].apply(
    lambda x: "zero" if x == 0 else ("low" if x <= 0.2 else "moderate_or_high")
)

print("Update volatility por entidade com rótulo simples:")
display(entity_update_volatility_df)

In [ ]:
# =========================================================
# BLOCK V16 — SUMMARIZE UPDATE VOLATILITY PER QUERY
# aligned to formulação metodológica (updavg, updmax)
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_compact_df",
    "entity_update_volatility_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Detectar coluna de entidades tocadas
# ---------------------------------------------------------
if "Qi_entities_touched" in analytical_matrix_compact_df.columns:
    touched_entities_col = "Qi_entities_touched"
elif "entities_touched" in analytical_matrix_compact_df.columns:
    touched_entities_col = "entities_touched"
else:
    raise ValueError(
        "analytical_matrix_compact_df não possui nem 'Qi_entities_touched' "
        "nem 'entities_touched'."
    )

# ---------------------------------------------------------
# 3) Create mapa entidade -> volatility
# ---------------------------------------------------------
entity_vol_map = dict(
    zip(
        entity_update_volatility_df["entity"],
        entity_update_volatility_df["update_volatility_score"]
    )
)

# ---------------------------------------------------------
# 4) Calcular volatilidade agregada por query
#    Equivalente à ideia metodológica de:
#    - updavg(Qi)
#    - updmax(Qi)
#    - número de entidades tocadas com volatility > 0
# ---------------------------------------------------------
def summarize_query_volatility(entities_touched):
    if not isinstance(entities_touched, list):
        entities_touched = []

    scores = [float(entity_vol_map.get(entity, 0.0)) for entity in entities_touched]

    return pd.Series({
        "updavg": float(np.mean(scores)) if scores else 0.0,
        "updmax": float(np.max(scores)) if scores else 0.0,
        "n_entities_with_nonzero_update_volatility": int(sum(1 for s in scores if s > 0))
    })

query_volatility_df = analytical_matrix_compact_df[
    ["query_name", touched_entities_col]
].copy()

query_volatility_df = query_volatility_df.rename(columns={
    touched_entities_col: "entities_touched_for_volatility"
})

volatility_summary_df = query_volatility_df["entities_touched_for_volatility"].apply(
    summarize_query_volatility
)

query_volatility_df = pd.concat(
    [query_volatility_df, volatility_summary_df],
    axis=1
)

print("Summary de update volatility por query:")
display(query_volatility_df)

# ---------------------------------------------------------
# 5) Enriquecer a matriz compacta com os resumos de volatility
# ---------------------------------------------------------
analytical_matrix_compact_with_volatility_df = analytical_matrix_compact_df.merge(
    query_volatility_df[
        [
            "query_name",
            "updavg",
            "updmax",
            "n_entities_with_nonzero_update_volatility"
        ]
    ],
    on="query_name",
    how="left"
)

print("\nMatriz compacta enriquecida com update volatility:")
display(analytical_matrix_compact_with_volatility_df)

In [ ]:
# =========================================================
# BLOCK V17 — INTEGRATE UPDATE VOLATILITY INTO THE ANALYTICAL MATRIX
# aligned to V16 atualizado
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_compact_df",
    "query_volatility_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Merge a volatilidade por query à matriz compacta
# ---------------------------------------------------------
analytical_matrix_with_volatility_df = analytical_matrix_compact_df.merge(
    query_volatility_df[
        [
            "query_name",
            "updavg",
            "updmax",
            "n_entities_with_nonzero_update_volatility"
        ]
    ],
    on="query_name",
    how="left"
)

# preencher ausências, se houver
for col in ["updavg", "updmax", "n_entities_with_nonzero_update_volatility"]:
    if col not in analytical_matrix_with_volatility_df.columns:
        analytical_matrix_with_volatility_df[col] = 0

analytical_matrix_with_volatility_df["updavg"] = (
    analytical_matrix_with_volatility_df["updavg"].fillna(0.0).astype(float)
)
analytical_matrix_with_volatility_df["updmax"] = (
    analytical_matrix_with_volatility_df["updmax"].fillna(0.0).astype(float)
)
analytical_matrix_with_volatility_df["n_entities_with_nonzero_update_volatility"] = (
    analytical_matrix_with_volatility_df["n_entities_with_nonzero_update_volatility"]
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 3) Create leitura qualitativa simples da volatilidade
# Esta classificação é interpretativa, não regra final.
# ---------------------------------------------------------
def classify_query_volatility(avg_vol):
    if avg_vol == 0:
        return "No volatility exposure"
    elif avg_vol <= 0.15:
        return "Low volatility exposure"
    elif avg_vol <= 0.30:
        return "Moderate volatility exposure"
    else:
        return "High volatility exposure"

analytical_matrix_with_volatility_df["query_volatility_class"] = (
    analytical_matrix_with_volatility_df["updavg"]
    .apply(classify_query_volatility)
)

# ---------------------------------------------------------
# 4) Sort para leitura
# ---------------------------------------------------------
sort_cols = [c for c in ["generic_class", "query_name"] if c in analytical_matrix_with_volatility_df.columns]
if sort_cols:
    analytical_matrix_with_volatility_df = analytical_matrix_with_volatility_df.sort_values(
        by=sort_cols
    ).reset_index(drop=True)
else:
    analytical_matrix_with_volatility_df = analytical_matrix_with_volatility_df.sort_values(
        by="query_name"
    ).reset_index(drop=True)

# ---------------------------------------------------------
# 5) Summary enxuto
# ---------------------------------------------------------
volatility_summary_cols = [
    c for c in [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "Rc",
        "D_value",
        "selected_document_depth",
        "selected_Re",
        "selected_DeltaR",
        "selected_DeltaR_ratio",
        "dominant_semantic_group",
        "dominant_semantic_detail",
        "updavg",
        "updmax",
        "n_entities_with_nonzero_update_volatility",
        "query_volatility_class"
    ]
    if c in analytical_matrix_with_volatility_df.columns
]

analytical_matrix_with_volatility_summary_df = analytical_matrix_with_volatility_df[
    volatility_summary_cols
].copy()

print("Compact analytical matrix for IMDb com update volatility:")
display(analytical_matrix_with_volatility_summary_df)

print("\nMatriz analítica detalhada com update volatility:")
display(analytical_matrix_with_volatility_df)

In [ ]:
# =========================================================
# BLOCK V18 — EXTRACT OBSERVED SHAREDNESS FROM THE IMDb DATASET
# aligned to IMDb real via DuckDB
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "imdb_data_bundle"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

con = imdb_data_bundle.con

# reduzir ruído no notebook
try:
    con.execute("PRAGMA disable_progress_bar")
except:
    pass

# ---------------------------------------------------------
# 2) Helper function para calcular sharedness observada
#
# shareobs(e | er)  = média de raízes distintas por instância target
# sharemax_obs(...) = máximo de raízes distintas por instância target
# ---------------------------------------------------------
def summarize_observed_sharedness_sql(
    con,
    rel_sql: str,
    root_col: str,
    target_col: str,
    root_entity: str,
    target_entity: str,
    relationship_context: str
) -> dict:
    sql = f"""
    WITH rel AS (
        SELECT DISTINCT
            {root_col} AS root_id,
            {target_col} AS target_id
        FROM ({rel_sql}) x
        WHERE {root_col} IS NOT NULL
          AND {target_col} IS NOT NULL
    ),
    target_to_roots AS (
        SELECT
            target_id,
            COUNT(DISTINCT root_id) AS n_roots
        FROM rel
        GROUP BY 1
    )
    SELECT
        '{root_entity}' AS root_entity,
        '{target_entity}' AS target_entity,
        '{relationship_context}' AS relationship_context,
        COUNT(*) AS n_target_instances_observed,
        ROUND(AVG(n_roots), 4) AS avg_roots_per_target,
        MAX(n_roots) AS max_roots_per_target
    FROM target_to_roots
    """
    return con.execute(sql).df().iloc[0].to_dict()

# ---------------------------------------------------------
# 3) Extrair sharedness para pares relevantes do cenário
# ---------------------------------------------------------
sharedness_rows = []

# WatchItem -> Role
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                watchitem_id,
                role_id
            FROM imdb_roles
        """,
        root_col="watchitem_id",
        target_col="role_id",
        root_entity="WatchItem",
        target_entity="Role",
        relationship_context="WatchItem to Role"
    )
)

# WatchItem -> Person (via Role)
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                watchitem_id,
                person_id
            FROM imdb_roles
        """,
        root_col="watchitem_id",
        target_col="person_id",
        root_entity="WatchItem",
        target_entity="Person",
        relationship_context="WatchItem to Person via Role"
    )
)

# WatchItem -> Genre
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                watchitem_id,
                primary_genre
            FROM imdb_watchitems
            WHERE primary_genre IS NOT NULL
        """,
        root_col="watchitem_id",
        target_col="primary_genre",
        root_entity="WatchItem",
        target_entity="Genre",
        relationship_context="WatchItem to Genre"
    )
)

# WatchItem -> Movie (subtipo)
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                watchitem_id AS root_id,
                watchitem_id AS target_id
            FROM imdb_movies
        """,
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Movie",
        relationship_context="Subtype identity: Movie is WatchItem"
    )
)

# WatchItem -> Series (subtipo)
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                watchitem_id AS root_id,
                watchitem_id AS target_id
            FROM imdb_series
        """,
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Series",
        relationship_context="Subtype identity: Series is WatchItem"
    )
)

# WatchItem -> Episode (subtipo)
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                episode_watchitem_id AS root_id,
                episode_watchitem_id AS target_id
            FROM imdb_episodes
        """,
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Episode",
        relationship_context="Subtype identity: Episode is WatchItem"
    )
)

# Series -> Episode (containment)
sharedness_rows.append(
    summarize_observed_sharedness_sql(
        con=con,
        rel_sql="""
            SELECT
                series_watchitem_id,
                episode_watchitem_id
            FROM imdb_episodes
            WHERE series_watchitem_id IS NOT NULL
        """,
        root_col="series_watchitem_id",
        target_col="episode_watchitem_id",
        root_entity="Series",
        target_entity="Episode",
        relationship_context="Series to Episode (containment)"
    )
)

sharedness_df = pd.DataFrame(sharedness_rows).sort_values(
    ["root_entity", "target_entity"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 4) Índices auxiliares para próximos blocos
# ---------------------------------------------------------
sharedness_map = {
    (row["root_entity"], row["target_entity"]): {
        "shareobs": row["avg_roots_per_target"],
        "sharemax_obs": row["max_roots_per_target"],
        "relationship_context": row["relationship_context"]
    }
    for _, row in sharedness_df.iterrows()
}

print("Sharedness observada no dataset IMDb real:")
display(sharedness_df)

In [ ]:
# =========================================================
# BLOCK V19 — SUMMARIZE SHAREDNESS PER QUERY
# aligned to V18 e à formulação metodológica
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_with_volatility_df",
    "sharedness_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Detectar coluna de entidades tocadas
# ---------------------------------------------------------
if "Qi_entities_touched" in analytical_matrix_with_volatility_df.columns:
    touched_entities_col = "Qi_entities_touched"
elif "entities_touched" in analytical_matrix_with_volatility_df.columns:
    touched_entities_col = "entities_touched"
else:
    raise ValueError(
        "analytical_matrix_with_volatility_df não possui nem "
        "'Qi_entities_touched' nem 'entities_touched'."
    )

# ---------------------------------------------------------
# 3) Create mapa (root_entity, target_entity) -> sharedness
# ---------------------------------------------------------
sharedness_map = {}

for _, row in sharedness_df.iterrows():
    key = (row["root_entity"], row["target_entity"])
    sharedness_map[key] = {
        "shareobs": float(row["avg_roots_per_target"]),
        "sharemax_obs": float(row["max_roots_per_target"]),
        "relationship_context": row["relationship_context"]
    }

# ---------------------------------------------------------
# 4) Helper function para resumir sharedness de uma query
#
# Regra metodológica:
# - excluir a própria raiz do cálculo agregado
# - agregar apenas entidades para as quais existe mapeamento observado
# ---------------------------------------------------------
def summarize_query_sharedness(selected_root, entities_touched):
    if not isinstance(entities_touched, list):
        entities_touched = []

    candidate_entities = [e for e in entities_touched if e != selected_root]

    avg_scores = []
    max_scores = []
    details = []

    for entity in candidate_entities:
        key = (selected_root, entity)
        info = sharedness_map.get(key, None)

        if info is not None:
            avg_scores.append(info["shareobs"])
            max_scores.append(info["sharemax_obs"])
            details.append({
                "target_entity": entity,
                "shareobs": info["shareobs"],
                "sharemax_obs": info["sharemax_obs"],
                "relationship_context": info["relationship_context"]
            })
        else:
            details.append({
                "target_entity": entity,
                "shareobs": None,
                "sharemax_obs": None,
                "relationship_context": "No observed mapping available"
            })

    return pd.Series({
        # nomes próximos à metodologia
        "shareavg": float(np.mean(avg_scores)) if avg_scores else np.nan,
        "sharemax": float(np.max(max_scores)) if max_scores else np.nan,
        "n_entities_with_sharedness_gt_1": int(sum(1 for x in avg_scores if x > 1)),
        "query_sharedness_details": details,

        # aliases compatíveis com leitura antiga
        "query_avg_sharedness": float(np.mean(avg_scores)) if avg_scores else np.nan,
        "query_max_sharedness": float(np.max(max_scores)) if max_scores else np.nan,
        "query_entities_with_sharedness_gt_1": int(sum(1 for x in avg_scores if x > 1))
    })

# ---------------------------------------------------------
# 5) Apply para cada query da matriz compacta
# ---------------------------------------------------------
query_sharedness_df = analytical_matrix_with_volatility_df[
    ["query_name", "selected_root", touched_entities_col]
].copy()

query_sharedness_df = query_sharedness_df.rename(columns={
    touched_entities_col: "entities_touched_for_sharedness"
})

sharedness_summary_df = query_sharedness_df.apply(
    lambda row: summarize_query_sharedness(
        selected_root=row["selected_root"],
        entities_touched=row["entities_touched_for_sharedness"]
    ),
    axis=1
)

query_sharedness_df = pd.concat([query_sharedness_df, sharedness_summary_df], axis=1)

# ---------------------------------------------------------
# 6) Classificação qualitativa simples
# limiares apenas interpretativos por enquanto
# ---------------------------------------------------------
def classify_sharedness(avg_sharedness):
    if pd.isna(avg_sharedness):
        return "Not applicable"
    elif avg_sharedness <= 1.0:
        return "Low sharedness"
    elif avg_sharedness <= 2.0:
        return "Moderate sharedness"
    else:
        return "High sharedness"

query_sharedness_df["query_sharedness_class"] = query_sharedness_df[
    "shareavg"
].apply(classify_sharedness)

# ---------------------------------------------------------
# 7) Display
# ---------------------------------------------------------
print("Summary de sharedness por query:")
display(query_sharedness_df)

# ---------------------------------------------------------
# 8) Integrar sharedness à matriz com volatility
# ---------------------------------------------------------
analytical_matrix_with_sharedness_df = analytical_matrix_with_volatility_df.merge(
    query_sharedness_df[
        [
            "query_name",
            "shareavg",
            "sharemax",
            "n_entities_with_sharedness_gt_1",
            "query_sharedness_class",
            "query_sharedness_details"
        ]
    ],
    on="query_name",
    how="left"
)

print("\nMatriz analítica compacta com volatility + sharedness:")
display(analytical_matrix_with_sharedness_df)

In [ ]:
# =========================================================
# BLOCK V20 — INTEGRATE SHAREDNESS INTO THE FINAL ANALYTICAL MATRIX
# aligned to V17 e V19 atualizados
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
has_premerged_matrix = "analytical_matrix_with_sharedness_df" in globals()
has_merge_inputs = (
    "analytical_matrix_with_volatility_df" in globals()
    and "query_sharedness_df" in globals()
)

if not has_premerged_matrix and not has_merge_inputs:
    raise NameError(
        "Nem 'analytical_matrix_with_sharedness_df' nem "
        "('analytical_matrix_with_volatility_df' + 'query_sharedness_df') "
        "estão disponíveis. Rode primeiro os blocos anteriores."
    )

# ---------------------------------------------------------
# 2) Base da matriz final
# ---------------------------------------------------------
if has_premerged_matrix:
    analytical_matrix_final_df = analytical_matrix_with_sharedness_df.copy()
else:
    # fallback: refaz o merge manualmente
    sharedness_cols_to_merge = [
        c for c in [
            "query_name",
            "shareavg",
            "sharemax",
            "n_entities_with_sharedness_gt_1",
            "query_sharedness_class",
            "query_sharedness_details"
        ]
        if c in query_sharedness_df.columns
    ]

    analytical_matrix_final_df = analytical_matrix_with_volatility_df.merge(
        query_sharedness_df[sharedness_cols_to_merge],
        on="query_name",
        how="left"
    )

# ---------------------------------------------------------
# 3) Compatibilizar nomes antigos/novos, se necessário
# ---------------------------------------------------------
if "query_avg_sharedness" not in analytical_matrix_final_df.columns and "shareavg" in analytical_matrix_final_df.columns:
    analytical_matrix_final_df["query_avg_sharedness"] = analytical_matrix_final_df["shareavg"]

if "query_max_sharedness" not in analytical_matrix_final_df.columns and "sharemax" in analytical_matrix_final_df.columns:
    analytical_matrix_final_df["query_max_sharedness"] = analytical_matrix_final_df["sharemax"]

if "query_entities_with_sharedness_gt_1" not in analytical_matrix_final_df.columns and "n_entities_with_sharedness_gt_1" in analytical_matrix_final_df.columns:
    analytical_matrix_final_df["query_entities_with_sharedness_gt_1"] = (
        analytical_matrix_final_df["n_entities_with_sharedness_gt_1"]
    )

# ---------------------------------------------------------
# 4) Detectar coluna de cobertura estrutural
# ---------------------------------------------------------
if "selected_structural_coverage" in analytical_matrix_final_df.columns:
    structural_coverage_col = "selected_structural_coverage"
elif "document_structural_coverage" in analytical_matrix_final_df.columns:
    structural_coverage_col = "document_structural_coverage"
else:
    raise ValueError(
        "A matriz final não possui nem 'selected_structural_coverage' "
        "nem 'document_structural_coverage'."
    )

# ---------------------------------------------------------
# 5) Create leitura qualitativa combinada
# Esta coluna é interpretativa por enquanto.
# ---------------------------------------------------------
def combined_document_readiness(row):
    coverage = row.get(structural_coverage_col, None)
    volatility = row.get("query_volatility_class", None)
    sharedness = row.get("query_sharedness_class", None)

    if coverage == "No structural traversal":
        return "Structurally neutral for document"

    if coverage == "Full absorption":
        if volatility == "No volatility exposure" and sharedness in ["Low sharedness", "Not applicable"]:
            return "Strong document candidate"
        elif volatility == "No volatility exposure" and sharedness in ["Moderate sharedness", "High sharedness"]:
            return "Promising document candidate with duplication trade-off"
        elif volatility in ["Low volatility exposure", "Moderate volatility exposure"] and sharedness in ["Low sharedness", "Moderate sharedness"]:
            return "Document candidate with maintenance trade-off"
        else:
            return "Document candidate with stronger trade-offs"

    if coverage == "Partial absorption":
        return "Partial document candidate"

    return "Weak document candidate"

analytical_matrix_final_df["document_candidate_assessment"] = analytical_matrix_final_df.apply(
    combined_document_readiness,
    axis=1
)

# ---------------------------------------------------------
# 6) Sort
# ---------------------------------------------------------
sort_cols = [c for c in ["generic_class", "query_name"] if c in analytical_matrix_final_df.columns]
if sort_cols:
    analytical_matrix_final_df = analytical_matrix_final_df.sort_values(
        by=sort_cols
    ).reset_index(drop=True)
else:
    analytical_matrix_final_df = analytical_matrix_final_df.sort_values(
        by="query_name"
    ).reset_index(drop=True)

# ---------------------------------------------------------
# 7) Summary enxuto
# ---------------------------------------------------------
final_summary_cols = [
    c for c in [
        "query_name",
        "generic_class",
        "query_type",
        "selected_root",
        "Rc",
        "D_value",
        "selected_document_depth",
        "selected_Re",
        "selected_DeltaR",
        "selected_DeltaR_ratio",
        structural_coverage_col,
        "dominant_semantic_group",
        "dominant_semantic_detail",
        "updavg",
        "updmax",
        "query_volatility_class",
        "shareavg",
        "sharemax",
        "query_sharedness_class",
        "document_candidate_assessment"
    ]
    if c in analytical_matrix_final_df.columns
]

analytical_matrix_final_summary_df = analytical_matrix_final_df[
    final_summary_cols
].copy()

print("Matriz analítica final do IMDb para o modelo documento:")
display(analytical_matrix_final_summary_df)

print("\nMatriz analítica final detalhada:")
display(analytical_matrix_final_df)

In [ ]:
#TRANSFORMAR MATRIZ ANALÓTICA EM CSV PARA ANÁLISE DOS RESULTADOS
analytical_matrix_final_df.to_csv("analytical_matrix_final_df.csv", index=False)

In [ ]:
# =========================================================
# BLOCK V21A — FINAL MATRIX OF CALCULATED VARIABLES
# adjusted to the current pipeline
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_final_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Base de trabalho
# ---------------------------------------------------------
base_df = analytical_matrix_final_df.copy()

# ---------------------------------------------------------
# 3) Helper function para escolher o primeiro nome disponível
# ---------------------------------------------------------
def pick_first_existing(df, candidates, required=False, default_value=None):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(
            f"Nenhuma das colunas esperadas foi encontrada: {candidates}"
        )
    return None

# ---------------------------------------------------------
# 4) Detectar nomes atuais das colunas
# ---------------------------------------------------------
entities_col = pick_first_existing(
    base_df,
    ["Qi_entities_touched", "entities_touched"],
    required=True
)

n_entities_col = pick_first_existing(
    base_df,
    ["Qi_n_entities_touched"],
    required=False
)

query_type_col = pick_first_existing(
    base_df,
    ["query_type", "type"],
    required=False
)

doc_depth_col = pick_first_existing(
    base_df,
    ["selected_document_depth", "document_depth_selected"],
    required=True
)

selected_re_col = pick_first_existing(
    base_df,
    ["selected_Re", "Re_selected"],
    required=True
)

selected_delta_r_col = pick_first_existing(
    base_df,
    ["selected_DeltaR", "DeltaR_selected"],
    required=True
)

selected_delta_r_ratio_col = pick_first_existing(
    base_df,
    ["selected_DeltaR_ratio", "selected_DeltaRratio", "DeltaRratio_selected"],
    required=True
)

coverage_col = pick_first_existing(
    base_df,
    ["selected_structural_coverage", "document_structural_coverage"],
    required=True
)

updavg_col = pick_first_existing(
    base_df,
    ["updavg", "query_avg_update_volatility"],
    required=False
)

updmax_col = pick_first_existing(
    base_df,
    ["updmax", "query_max_update_volatility"],
    required=False
)

updcount_col = pick_first_existing(
    base_df,
    ["n_entities_with_nonzero_update_volatility", "query_entities_with_nonzero_volatility"],
    required=False
)

shareavg_col = pick_first_existing(
    base_df,
    ["shareavg", "query_avg_sharedness"],
    required=False
)

sharemax_col = pick_first_existing(
    base_df,
    ["sharemax", "query_max_sharedness"],
    required=False
)

sharecount_col = pick_first_existing(
    base_df,
    ["n_entities_with_sharedness_gt_1", "query_entities_with_sharedness_gt_1"],
    required=False
)

volatility_class_col = pick_first_existing(
    base_df,
    ["query_volatility_class"],
    required=False
)

sharedness_class_col = pick_first_existing(
    base_df,
    ["query_sharedness_class"],
    required=False
)

assessment_col = pick_first_existing(
    base_df,
    ["document_candidate_assessment"],
    required=False
)

abstract_query_col = pick_first_existing(
    base_df,
    ["abstract_query"],
    required=False
)

# ---------------------------------------------------------
# 5) Create aliases padronizados para a matriz final
# ---------------------------------------------------------
document_variable_matrix_df = pd.DataFrame()

document_variable_matrix_df["query_name"] = base_df["query_name"]
document_variable_matrix_df["query_type"] = base_df[query_type_col] if query_type_col else None
document_variable_matrix_df["Qi_entities_touched"] = base_df[entities_col]

if n_entities_col:
    document_variable_matrix_df["Qi_n_entities_touched"] = base_df[n_entities_col]
else:
    document_variable_matrix_df["Qi_n_entities_touched"] = base_df[entities_col].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

document_variable_matrix_df["abstract_query"] = base_df[abstract_query_col] if abstract_query_col else None

# núcleo estrutural
document_variable_matrix_df["selected_root"] = base_df["selected_root"]
document_variable_matrix_df["Rc"] = base_df["Rc"]
document_variable_matrix_df["D_value"] = base_df["D_value"]
document_variable_matrix_df["selected_document_depth"] = base_df[doc_depth_col]
document_variable_matrix_df["selected_Re"] = base_df[selected_re_col]
document_variable_matrix_df["selected_DeltaR"] = base_df[selected_delta_r_col]
document_variable_matrix_df["selected_DeltaR_ratio"] = base_df[selected_delta_r_ratio_col]
document_variable_matrix_df["selected_structural_coverage"] = base_df[coverage_col]

# perfil semântico principal
for col in ["n_association", "n_associative", "n_containment"]:
    if col in base_df.columns:
        document_variable_matrix_df[col] = base_df[col]
    else:
        document_variable_matrix_df[col] = 0

# volatilidade
document_variable_matrix_df["query_avg_update_volatility"] = base_df[updavg_col] if updavg_col else 0.0
document_variable_matrix_df["query_max_update_volatility"] = base_df[updmax_col] if updmax_col else 0.0
document_variable_matrix_df["query_entities_with_nonzero_volatility"] = base_df[updcount_col] if updcount_col else 0
document_variable_matrix_df["query_volatility_class"] = base_df[volatility_class_col] if volatility_class_col else None

# sharedness
document_variable_matrix_df["query_avg_sharedness"] = base_df[shareavg_col] if shareavg_col else None
document_variable_matrix_df["query_max_sharedness"] = base_df[sharemax_col] if sharemax_col else None
document_variable_matrix_df["query_entities_with_sharedness_gt_1"] = base_df[sharecount_col] if sharecount_col else 0
document_variable_matrix_df["query_sharedness_class"] = base_df[sharedness_class_col] if sharedness_class_col else None

# leitura interpretativa
document_variable_matrix_df["document_candidate_assessment"] = base_df[assessment_col] if assessment_col else None

# ---------------------------------------------------------
# 6) Colunas auxiliares para leitura
# ---------------------------------------------------------
document_variable_matrix_df["has_full_absorption"] = (
    document_variable_matrix_df["selected_structural_coverage"] == "Full absorption"
)

document_variable_matrix_df["is_structurally_neutral"] = (
    document_variable_matrix_df["selected_structural_coverage"] == "No structural traversal"
)

# auxiliares metodológicas úteis
document_variable_matrix_df["has_any_association"] = (
    document_variable_matrix_df["n_association"] > 0
)

document_variable_matrix_df["has_any_associative"] = (
    document_variable_matrix_df["n_associative"] > 0
)

document_variable_matrix_df["has_any_containment"] = (
    document_variable_matrix_df["n_containment"] > 0
)

# ---------------------------------------------------------
# 7) Sort
# ---------------------------------------------------------
document_variable_matrix_df = document_variable_matrix_df.sort_values(
    by="query_name"
).reset_index(drop=True)

print("Matriz final de variáveis calculadas do IMDb:")
display(document_variable_matrix_df)

In [ ]:
# =========================================================
# BLOCK V21B — INFERRED ACTIVATION OF CLASSES AND GENERATION
# OF CANDIDATE CONFIGURATIONS FOR TESTING
# guided by the computed matrix, with no hardcoding from the old queries
# =========================================================

import pandas as pd
import re

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "document_variable_matrix_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V21A."
        )

# optional, but very useful for concrete inference
optional_names = [
    "analytical_matrix_final_df",
    "query_relationship_semantics_df"
]

for name in optional_names:
    if name not in globals():
        print(f"Warning: '{name}' is not defined. O bloco still runs, but with less rich inference.")

# ---------------------------------------------------------
# 2) Main base
# ---------------------------------------------------------
activation_base_df = document_variable_matrix_df.copy()

# bring additional columns if they exist in the detailed matrix
if "analytical_matrix_final_df" in globals():
    extra_cols = [
        c for c in [
            "query_name",
            "generic_class",
            "dominant_semantic_group",
            "dominant_semantic_detail",
            "n_shared_descriptor_edges",
            "n_structural_component_edges",
            "n_associative_bridge_edges",
            "n_containment_edges",
            "query_sharedness_details",
            "entity_distances_from_root",
            "Rc_selected_edges"
        ]
        if c in analytical_matrix_final_df.columns
    ]

    activation_base_df = activation_base_df.merge(
        analytical_matrix_final_df[extra_cols].drop_duplicates(subset=["query_name"]),
        on="query_name",
        how="left"
    )

# se generic_class ainda não existir, tenta inferir do nome da query
if "generic_class" not in activation_base_df.columns:
    def infer_generic_class_from_query_name(query_name):
        m = re.match(r"^(QG\d+)", str(query_name))
        return m.group(1) if m else None

    activation_base_df["generic_class"] = activation_base_df["query_name"].apply(
        infer_generic_class_from_query_name
    )

# ---------------------------------------------------------
# 3) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def low_volatility(row):
    avg = row.get("query_avg_update_volatility", 0.0)
    cls = row.get("query_volatility_class", "")
    return (avg == 0) or (avg <= 0.15) or (cls in ["No volatility exposure", "Low volatility exposure"])

def low_or_moderate_volatility(row):
    avg = row.get("query_avg_update_volatility", 0.0)
    cls = row.get("query_volatility_class", "")
    return (avg <= 0.30) or (cls in ["No volatility exposure", "Low volatility exposure", "Moderate volatility exposure"])

def low_sharedness(row):
    avg = row.get("query_avg_sharedness", None)
    cls = row.get("query_sharedness_class", "")
    if pd.isna(avg):
        return True
    return (avg <= 1.0) or (cls in ["Low sharedness", "Not applicable"])

def moderate_or_high_sharedness(row):
    avg = row.get("query_avg_sharedness", None)
    cls = row.get("query_sharedness_class", "")
    if pd.isna(avg):
        return False
    return (avg > 1.0) or (cls in ["Moderate sharedness", "High sharedness"])

def has_full_absorption(row):
    return row.get("selected_structural_coverage", "") == "Full absorption"

def has_partial_absorption(row):
    return row.get("selected_structural_coverage", "") == "Partial absorption"

def get_relationship_rows_for_query(query_name):
    if "query_relationship_semantics_df" not in globals():
        return pd.DataFrame()
    return query_relationship_semantics_df[
        query_relationship_semantics_df["query_name"] == query_name
    ].copy()

def infer_entities_by_semantic_detail(query_name, selected_root, semantic_detail):
    """
    Extrai entidades candidatas a partir das arestas tocadas pela query,
    usando semantic_detail e removendo a raiz.
    """
    rel_df = get_relationship_rows_for_query(query_name)
    if rel_df.empty:
        return []

    entities = set()

    for _, r in rel_df.iterrows():
        if r.get("semantic_detail", None) != semantic_detail:
            continue

        edge_id = str(r["edge_id"])
        # formato esperado: "A -- B [rel]"
        edge_main = edge_id.split(" [")[0]
        if " -- " not in edge_main:
            continue

        src, tgt = edge_main.split(" -- ", 1)

        if src != selected_root:
            entities.add(src)
        if tgt != selected_root:
            entities.add(tgt)

    return sorted(list(entities))

def infer_contained_entities(query_name, selected_root):
    rel_df = get_relationship_rows_for_query(query_name)
    if rel_df.empty:
        return []

    entities = set()

    for _, r in rel_df.iterrows():
        if r.get("semantic_group", None) != "containment":
            continue

        edge_id = str(r["edge_id"])
        edge_main = edge_id.split(" [")[0]
        if " -- " not in edge_main:
            continue

        src, tgt = edge_main.split(" -- ", 1)

        if src != selected_root:
            entities.add(src)
        if tgt != selected_root:
            entities.add(tgt)

    return sorted(list(entities))

def infer_associative_entities(row):
    """
    Inferência guiada por:
    - arestas associativas tocadas pela query
    - distâncias a partir da raiz
    """
    query_name = row["query_name"]
    selected_root = row["selected_root"]
    rel_df = get_relationship_rows_for_query(query_name)

    if rel_df.empty:
        return [], []

    assoc_df = rel_df[rel_df["semantic_group"] == "associative"].copy()
    if assoc_df.empty:
        return [], []

    entity_distances = row.get("entity_distances_from_root", {})
    if not isinstance(entity_distances, dict):
        entity_distances = {}

    candidates = set()

    for _, r in assoc_df.iterrows():
        edge_id = str(r["edge_id"])
        edge_main = edge_id.split(" [")[0]
        if " -- " not in edge_main:
            continue
        src, tgt = edge_main.split(" -- ", 1)

        if src != selected_root:
            candidates.add(src)
        if tgt != selected_root:
            candidates.add(tgt)

    if not candidates:
        return [], []

    # bridge = menor distância positiva até a raiz
    # final  = maior distância
    candidate_rows = []
    for ent in candidates:
        d = entity_distances.get(ent, None)
        if d is not None:
            candidate_rows.append((ent, d))

    if not candidate_rows:
        return sorted(list(candidates)), []

    positive_rows = [(e, d) for e, d in candidate_rows if d > 0]
    if not positive_rows:
        return [], []

    min_d = min(d for _, d in positive_rows)
    max_d = max(d for _, d in positive_rows)

    bridge_entities = sorted([e for e, d in positive_rows if d == min_d])
    final_entities = sorted([e for e, d in positive_rows if d == max_d and e not in bridge_entities])

    return bridge_entities, final_entities

def normalize_entity_list(values):
    seen = []
    for v in values:
        if v is None:
            continue
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

# ---------------------------------------------------------
# 4) Ativação inferida por query
# ---------------------------------------------------------
activation_rows = []

for _, row in activation_base_df.iterrows():
    query_name = row["query_name"]
    generic_class = row.get("generic_class", None)
    selected_root = row["selected_root"]
    rc = row["Rc"]
    d_value = row["D_value"]
    selected_document_depth = row["selected_document_depth"]
    n_association = row.get("n_association", 0)
    n_associative = row.get("n_associative", 0)
    n_containment = row.get("n_containment", 0)

    n_shared_descriptor_edges = row.get("n_shared_descriptor_edges", 0)
    n_structural_component_edges = row.get("n_structural_component_edges", 0)
    n_associative_bridge_edges = row.get("n_associative_bridge_edges", 0)
    n_containment_edges = row.get("n_containment_edges", 0)

    # -----------------------------
    # G0 — baseline documental
    # baseline principal para raízes não-containment
    # -----------------------------
    if n_containment == 0:
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G0",
            "selected_root": selected_root,
            "activation_reason": (
                "Baseline documental principal para a raiz selecionada."
            )
        })

    # -----------------------------
    # G1 — embutimento raso local
    # -----------------------------
    if (
        n_association > 0
        and d_value == 1
        and has_full_absorption(row)
        and low_volatility(row)
        and low_sharedness(row)
        and n_shared_descriptor_edges == 0
        and n_structural_component_edges == 0
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G1",
            "selected_root": selected_root,
            "activation_reason": (
                "Há ganho raso limpo com associação local, baixa volatilidade "
                "e baixa sharedness."
            )
        })

    # -----------------------------
    # G2 — embutimento de descritor compartilhado
    # -----------------------------
    if (
        n_association > 0
        and d_value == 1
        and has_full_absorption(row)
        and n_shared_descriptor_edges > 0
        and low_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G2",
            "selected_root": selected_root,
            "activation_reason": (
                "A matriz indica descritor compartilhado próximo da raiz, "
                "com absorção total rasa e baixa volatilidade."
            )
        })

    # -----------------------------
    # G3 — subtipo/componente estrutural próximo
    # -----------------------------
    if (
        n_association > 0
        and d_value == 1
        and has_full_absorption(row)
        and n_structural_component_edges > 0
        and low_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G3",
            "selected_root": selected_root,
            "activation_reason": (
                "A matriz indica subtipo/componente estrutural próximo da raiz, "
                "com ganho raso e baixa volatilidade."
            )
        })

    # -----------------------------
    # G4 — associativa parcial
    # -----------------------------
    if (
        n_associative > 0
        and n_associative_bridge_edges > 0
        and low_or_moderate_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G4",
            "selected_root": selected_root,
            "activation_reason": (
                "A matriz indica região associativa relevante; a entidade "
                "associativa próxima da raiz é candidata a embutimento parcial."
            )
        })

    # -----------------------------
    # G5 — associativa com snapshot
    # -----------------------------
    if (
        n_associative > 0
        and d_value >= 2
        and moderate_or_high_sharedness(row)
        and low_or_moderate_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G5",
            "selected_root": selected_root,
            "activation_reason": (
                "Há caminho associativo profundo com entidade final compartilhada; "
                "snapshot é candidato quando full embedding pode ser arriscado."
            )
        })

    # -----------------------------
    # G6 — associativa full
    # -----------------------------
    if (
        n_associative > 0
        and d_value >= 2
        and low_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G6",
            "selected_root": selected_root,
            "activation_reason": (
                "Há caminho associativo profundo e baixa volatilidade suficiente "
                "para testar o limite superior da absorção."
            )
        })

    # -----------------------------
    # G7 — baseline de containment
    # -----------------------------
    if n_containment > 0:
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G7",
            "selected_root": selected_root,
            "activation_reason": (
                "Baseline da família de containment para a raiz de composição."
            )
        })

    # -----------------------------
    # G8 — containment reduzido
    # -----------------------------
    if (
        n_containment > 0
        and d_value >= 1
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G8",
            "selected_root": selected_root,
            "activation_reason": (
                "Há estrutura parte-todo com componente contido que pode ser "
                "embutido de forma reduzida."
            )
        })

    # -----------------------------
    # G9 — containment completo
    # -----------------------------
    if (
        n_containment > 0
        and d_value >= 1
        and low_sharedness(row)
        and low_volatility(row)
    ):
        activation_rows.append({
            "query_name": query_name,
            "generic_class": generic_class,
            "activated_class": "G9",
            "selected_root": selected_root,
            "activation_reason": (
                "Há containment forte e estável; o embutimento completo do "
                "componente contido é candidato."
            )
        })

query_class_activation_df = pd.DataFrame(activation_rows)

# remover duplicatas eventuais
query_class_activation_df = query_class_activation_df.drop_duplicates(
    subset=["query_name", "activated_class", "selected_root"]
).sort_values(
    ["query_name", "activated_class"]
).reset_index(drop=True)

print("Classes ativadas por query (inferência guiada pela matriz):")
display(query_class_activation_df)

# ---------------------------------------------------------
# 5) Instanciação concreta das classes ativadas
#    -> gera catálogo de configurações candidatas
# ---------------------------------------------------------
config_rows = []

for _, act in query_class_activation_df.iterrows():
    query_name = act["query_name"]
    activated_class = act["activated_class"]
    selected_root = act["selected_root"]

    row = activation_base_df[
        activation_base_df["query_name"] == query_name
    ].iloc[0]

    entities_touched = as_list(row.get("Qi_entities_touched", row.get("entities_touched", [])))

    descriptor_entities = infer_entities_by_semantic_detail(
        query_name=query_name,
        selected_root=selected_root,
        semantic_detail="descriptive_association"
    )

    structural_entities = infer_entities_by_semantic_detail(
        query_name=query_name,
        selected_root=selected_root,
        semantic_detail="structural_association"
    )

    contained_entities = infer_contained_entities(
        query_name=query_name,
        selected_root=selected_root
    )

    bridge_entities, final_associative_entities = infer_associative_entities(row)

    # defaults
    proposed_embedding_depth = 0
    embedded_entities = []
    snapshot_entities = []
    referenced_entities = []
    embedding_variant = "baseline"

    if activated_class == "G0":
        proposed_embedding_depth = 0
        embedded_entities = []
        referenced_entities = [e for e in entities_touched if e != selected_root]
        embedding_variant = "baseline_document"

    elif activated_class == "G1":
        proposed_embedding_depth = 1
        embedded_entities = []
        referenced_entities = [e for e in entities_touched if e != selected_root]
        embedding_variant = "shallow_local"

    elif activated_class == "G2":
        proposed_embedding_depth = 1
        embedded_entities = descriptor_entities
        referenced_entities = [e for e in entities_touched if e != selected_root and e not in embedded_entities]
        embedding_variant = "shared_descriptor"

    elif activated_class == "G3":
        proposed_embedding_depth = 1
        embedded_entities = structural_entities
        referenced_entities = [e for e in entities_touched if e != selected_root and e not in embedded_entities]
        embedding_variant = "structural_component"

    elif activated_class == "G4":
        proposed_embedding_depth = 1
        embedded_entities = bridge_entities
        referenced_entities = [e for e in entities_touched if e != selected_root and e not in embedded_entities]
        embedding_variant = "associative_partial"

    elif activated_class == "G5":
        proposed_embedding_depth = max(2, int(row["D_value"]))
        embedded_entities = bridge_entities
        snapshot_entities = final_associative_entities
        referenced_entities = [
            e for e in entities_touched
            if e != selected_root and e not in embedded_entities
        ]
        embedding_variant = "associative_snapshot"

    elif activated_class == "G6":
        proposed_embedding_depth = max(2, int(row["D_value"]))
        embedded_entities = normalize_entity_list([bridge_entities, final_associative_entities])
        referenced_entities = [
            e for e in entities_touched
            if e != selected_root and e not in embedded_entities
        ]
        embedding_variant = "associative_full"

    elif activated_class == "G7":
        proposed_embedding_depth = 0
        embedded_entities = []
        referenced_entities = [e for e in entities_touched if e != selected_root]
        embedding_variant = "containment_baseline"

    elif activated_class == "G8":
        proposed_embedding_depth = 1
        embedded_entities = contained_entities
        referenced_entities = [e for e in entities_touched if e != selected_root and e not in embedded_entities]
        embedding_variant = "containment_reduced"

    elif activated_class == "G9":
        proposed_embedding_depth = 1
        embedded_entities = contained_entities
        referenced_entities = [e for e in entities_touched if e != selected_root and e not in embedded_entities]
        embedding_variant = "containment_full"

    # id e nome da configuração
    config_id = f"CFG_{selected_root}_{activated_class}_{query_name}"
    config_name = f"{selected_root.lower()}_{activated_class.lower()}"

    config_rows.append({
        "config_id": config_id,
        "config_name": config_name,
        "activated_class": activated_class,
        "derived_from_query": query_name,
        "generic_class": act["generic_class"],
        "selected_root": selected_root,
        "proposed_embedding_depth": proposed_embedding_depth,
        "embedded_entities": normalize_entity_list(embedded_entities),
        "snapshot_entities": normalize_entity_list(snapshot_entities),
        "referenced_entities": normalize_entity_list(referenced_entities),
        "embedding_variant": embedding_variant,
        "activation_reason": act["activation_reason"]
    })

mongo_configuration_candidates_df = pd.DataFrame(config_rows)

# ---------------------------------------------------------
# 6) Catálogo agregado de configurações
#    agrupa ativações repetidas da mesma forma estrutural
# ---------------------------------------------------------
if not mongo_configuration_candidates_df.empty:
    grouped_rows = []

    group_cols = [
        "config_name",
        "activated_class",
        "selected_root",
        "proposed_embedding_depth",
        "embedding_variant"
    ]

    for keys, grp in mongo_configuration_candidates_df.groupby(group_cols):
        config_name, activated_class, selected_root, proposed_embedding_depth, embedding_variant = keys

        all_embedded = normalize_entity_list(grp["embedded_entities"].tolist())
        all_snapshots = normalize_entity_list(grp["snapshot_entities"].tolist())
        all_referenced = normalize_entity_list(grp["referenced_entities"].tolist())
        all_queries = sorted(grp["derived_from_query"].unique().tolist())

        grouped_rows.append({
            "config_name": config_name,
            "activated_class": activated_class,
            "selected_root": selected_root,
            "proposed_embedding_depth": proposed_embedding_depth,
            "embedding_variant": embedding_variant,
            "embedded_entities": all_embedded,
            "snapshot_entities": all_snapshots,
            "referenced_entities": all_referenced,
            "derived_from_queries": all_queries,
            "n_supporting_queries": len(all_queries)
        })

    mongo_configuration_catalog_df = pd.DataFrame(grouped_rows).sort_values(
        ["selected_root", "activated_class", "config_name"]
    ).reset_index(drop=True)
else:
    mongo_configuration_catalog_df = pd.DataFrame()

# ---------------------------------------------------------
# 7) Versões amigáveis para visualização
# ---------------------------------------------------------
query_class_activation_view_df = query_class_activation_df.copy()

mongo_configuration_candidates_view_df = mongo_configuration_candidates_df.copy()
for col in ["embedded_entities", "snapshot_entities", "referenced_entities"]:
    if col in mongo_configuration_candidates_view_df.columns:
        mongo_configuration_candidates_view_df[col] = mongo_configuration_candidates_view_df[col].apply(list_to_str)

mongo_configuration_catalog_view_df = mongo_configuration_catalog_df.copy()
for col in ["embedded_entities", "snapshot_entities", "referenced_entities", "derived_from_queries"]:
    if col in mongo_configuration_catalog_view_df.columns:
        mongo_configuration_catalog_view_df[col] = mongo_configuration_catalog_view_df[col].apply(list_to_str)

print("Ativação inferida das classes por query:")
display(query_class_activation_view_df)

print("\nConfigurações candidatas geradas por ativação:")
display(mongo_configuration_candidates_view_df)

print("\nCatálogo agregado das principais configurações candidatas para implementação/teste:")
display(mongo_configuration_catalog_view_df)

In [ ]:
# =========================================================
# BLOCK V22 — EXPANDED CATALOG OF MONGODB EXPERIMENTS
# adjusted to the catalog inferred in V21B and with coverage
# of benchmark: primary, secondary affected, and control
# =========================================================

import pandas as pd
import re

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_configuration_catalog_df",
    "document_variable_matrix_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos V21A e V21B."
        )

# ---------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def normalize_entity_list(values):
    seen = []
    for v in values:
        if v is None:
            continue
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

def infer_generic_class(query_name):
    m = re.match(r"^(QG\d+)", str(query_name))
    return m.group(1) if m else None

def family_from_activated_class(activated_class):
    if activated_class in ["G0", "G1", "G2", "G3"]:
        return "root_local_family"
    if activated_class in ["G4", "G5", "G6"]:
        return "associative_family"
    if activated_class in ["G7", "G8", "G9"]:
        return "containment_family"
    return "other_family"

def primary_collection_from_root(root):
    if root == "WatchItem":
        return "watchitems"
    if root == "Series":
        return "series"
    return str(root).lower()

def safe_generic_class_col(df):
    if "generic_class" in df.columns:
        return "generic_class"
    return None

# ---------------------------------------------------------
# 3) Base de queries do workload final
# ---------------------------------------------------------
query_base_df = document_variable_matrix_df.copy()

if "Qi_entities_touched" not in query_base_df.columns:
    if "entities_touched" in query_base_df.columns:
        query_base_df["Qi_entities_touched"] = query_base_df["entities_touched"]
    else:
        raise KeyError(
            "document_variable_matrix_df não possui nem 'Qi_entities_touched' "
            "nem 'entities_touched'."
        )

if "generic_class" not in query_base_df.columns:
    query_base_df["generic_class"] = query_base_df["query_name"].apply(infer_generic_class)

if "selected_root" not in query_base_df.columns:
    raise KeyError("document_variable_matrix_df precisa ter 'selected_root'.")

# mapa de consulta
query_info_map = {}
for _, row in query_base_df.iterrows():
    query_info_map[row["query_name"]] = {
        "generic_class": row.get("generic_class", infer_generic_class(row["query_name"])),
        "entities_touched": as_list(row["Qi_entities_touched"]),
        "selected_root": row["selected_root"],
        "query_type": row.get("query_type", None),
        "Rc": row.get("Rc", None),
        "D_value": row.get("D_value", None),
    }

all_query_names = sorted(query_info_map.keys())

# ---------------------------------------------------------
# 4) Scale factors do estudo atual
# ---------------------------------------------------------
mongo_scale_factors_df = pd.DataFrame([
    {
        "scale_factor": 0.25,
        "scale_label": "sf0.25",
        "execution_phase": "included_in_study"
    },
    {
        "scale_factor": 0.50,
        "scale_label": "sf0.5",
        "execution_phase": "included_in_study"
    },
    {
        "scale_factor": 1.00,
        "scale_label": "sf1",
        "execution_phase": "included_in_study"
    }
])

print("Scale factors do estudo atual:")
display(mongo_scale_factors_df)

# ---------------------------------------------------------
# 5) Inferir cobertura do benchmark por configuração
#
# Regra geral:
# - primárias: queries que ativaram/sustentaram a configuração
# - secundárias afetadas: queries fora das primárias, mas que
#   tocam a região estrutural alterada pela configuração
# - controle: queries fora das primárias/secundárias, com baixa
#   ou nenhuma sobreposição estrutural
# ---------------------------------------------------------
coverage_rows = []

for _, cfg in mongo_configuration_catalog_df.iterrows():
    config_name = cfg["config_name"]
    activated_class = cfg["activated_class"]
    selected_root = cfg["selected_root"]
    proposed_embedding_depth = cfg["proposed_embedding_depth"]
    embedding_variant = cfg["embedding_variant"]

    embedded_entities = normalize_entity_list(cfg.get("embedded_entities", []))
    snapshot_entities = normalize_entity_list(cfg.get("snapshot_entities", []))
    referenced_entities_raw = normalize_entity_list(cfg.get("referenced_entities", []))
    derived_from_queries = normalize_entity_list(cfg.get("derived_from_queries", []))

    # corrigir referenced para não duplicar entidades já embutidas/snapshot
    referenced_entities = [
        e for e in referenced_entities_raw
        if e not in embedded_entities and e not in snapshot_entities
    ]

    family = family_from_activated_class(activated_class)

    # região estrutural modificada pela configuração
    changed_region_entities = normalize_entity_list(
        [selected_root, embedded_entities, snapshot_entities]
    )

    primary_queries = sorted(derived_from_queries)
    secondary_affected_queries = []
    control_queries = []

    for qname in all_query_names:
        if qname in primary_queries:
            continue

        qinfo = query_info_map[qname]
        q_entities = set(qinfo["entities_touched"])

        overlap_changed_region = len(q_entities.intersection(set(changed_region_entities)))
        touches_root = selected_root in q_entities

        # Regras de inferência por família
        if family == "root_local_family":
            # mudanças rasas/localizadas em torno da raiz
            if overlap_changed_region > 0 or touches_root:
                secondary_affected_queries.append(qname)
            else:
                control_queries.append(qname)

        elif family == "associative_family":
            # impacto mais forte em root + ponte associativa + entidade final
            if overlap_changed_region > 0 or touches_root:
                secondary_affected_queries.append(qname)
            else:
                control_queries.append(qname)

        elif family == "containment_family":
            # impacto principal em raiz de containment e componente contido
            if overlap_changed_region > 0:
                secondary_affected_queries.append(qname)
            else:
                control_queries.append(qname)

        else:
            if overlap_changed_region > 0:
                secondary_affected_queries.append(qname)
            else:
                control_queries.append(qname)

    # ordenar e deduplicar
    secondary_affected_queries = sorted(list(dict.fromkeys(secondary_affected_queries)))
    control_queries = sorted(list(dict.fromkeys(control_queries)))

    coverage_rows.append({
        "config_name": config_name,
        "activated_class": activated_class,
        "selected_root": selected_root,
        "embedding_variant": embedding_variant,
        "benchmark_family": family,
        "primary_queries": primary_queries,
        "secondary_affected_queries": secondary_affected_queries,
        "control_queries": control_queries,
        "n_primary_queries": len(primary_queries),
        "n_secondary_affected_queries": len(secondary_affected_queries),
        "n_control_queries": len(control_queries),
        "changed_region_entities": changed_region_entities
    })

benchmark_coverage_df = pd.DataFrame(coverage_rows)

print("Cobertura do benchmark por configuração:")
display(benchmark_coverage_df)

# ---------------------------------------------------------
# 6) Expandir cada configuração para cada scale factor
#    one row per (configuração, scale factor)
# ---------------------------------------------------------
mongo_experiment_rows = []

for _, cfg in mongo_configuration_catalog_df.iterrows():
    cfg_name = cfg["config_name"]
    cov_row = benchmark_coverage_df[
        benchmark_coverage_df["config_name"] == cfg_name
    ].iloc[0]

    activated_class = cfg["activated_class"]
    selected_root = cfg["selected_root"]
    proposed_embedding_depth = cfg["proposed_embedding_depth"]
    embedding_variant = cfg["embedding_variant"]

    embedded_entities = normalize_entity_list(cfg.get("embedded_entities", []))
    snapshot_entities = normalize_entity_list(cfg.get("snapshot_entities", []))
    referenced_entities_raw = normalize_entity_list(cfg.get("referenced_entities", []))
    referenced_entities = [
        e for e in referenced_entities_raw
        if e not in embedded_entities and e not in snapshot_entities
    ]
    derived_from_queries = normalize_entity_list(cfg.get("derived_from_queries", []))

    for _, sf_row in mongo_scale_factors_df.iterrows():
        scale_factor = sf_row["scale_factor"]
        scale_label = sf_row["scale_label"]
        execution_phase = sf_row["execution_phase"]

        mongo_db_name = f"imdb_mongo_{scale_label}_{cfg_name}"

        mongo_experiment_rows.append({
            "experiment_id": f"{cfg_name}_{scale_label}",
            "config_name": cfg_name,
            "activated_class": activated_class,
            "benchmark_family": cov_row["benchmark_family"],
            "scale_factor": scale_factor,
            "scale_label": scale_label,
            "execution_phase": execution_phase,
            "mongo_db_name": mongo_db_name,

            # estrutura documental
            "selected_root": selected_root,
            "primary_collection": primary_collection_from_root(selected_root),
            "embedding_depth": proposed_embedding_depth,
            "embedding_variant": embedding_variant,
            "embedded_entities": embedded_entities,
            "snapshot_entities": snapshot_entities,
            "referenced_entities": referenced_entities,
            "changed_region_entities": normalize_entity_list(cov_row["changed_region_entities"]),

            # motivação
            "derived_from_queries": derived_from_queries,
            "n_supporting_queries": cfg["n_supporting_queries"],

            # cobertura do benchmark
            "primary_queries": normalize_entity_list(cov_row["primary_queries"]),
            "secondary_affected_queries": normalize_entity_list(cov_row["secondary_affected_queries"]),
            "control_queries": normalize_entity_list(cov_row["control_queries"]),
            "n_primary_queries": cov_row["n_primary_queries"],
            "n_secondary_affected_queries": cov_row["n_secondary_affected_queries"],
            "n_control_queries": cov_row["n_control_queries"],

            # objetivo do experimento
            "experiment_goal": (
                f"Testar a configuração {cfg_name} ({activated_class}) "
                f"na família {cov_row['benchmark_family']} em {scale_label}, "
                f"avaliando queries primárias, secundárias afetadas e de controle."
            )
        })

mongo_experiment_catalog_df = pd.DataFrame(mongo_experiment_rows)

# ---------------------------------------------------------
# 7) Versões amigáveis para visualização
# ---------------------------------------------------------
benchmark_coverage_view_df = benchmark_coverage_df.copy()
for col in [
    "primary_queries",
    "secondary_affected_queries",
    "control_queries",
    "changed_region_entities"
]:
    benchmark_coverage_view_df[col] = benchmark_coverage_view_df[col].apply(list_to_str)

mongo_experiment_catalog_view_df = mongo_experiment_catalog_df.copy()
for col in [
    "embedded_entities",
    "snapshot_entities",
    "referenced_entities",
    "changed_region_entities",
    "derived_from_queries",
    "primary_queries",
    "secondary_affected_queries",
    "control_queries"
]:
    mongo_experiment_catalog_view_df[col] = mongo_experiment_catalog_view_df[col].apply(list_to_str)

# ---------------------------------------------------------
# 8) Display catálogos
# ---------------------------------------------------------
print("Cobertura do benchmark por configuração (primárias, secundárias, controle):")
display(benchmark_coverage_view_df)

print("\nExpanded catalog of MongoDB experiments:")
display(mongo_experiment_catalog_view_df)

In [ ]:
#TEST EXECUTION ORDER

# =========================================================
# BLOCK V22B — SUMMARY TABLE WITH THE ORDER OF EXPERIMENTS
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_experiment_catalog_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V22."
        )

# ---------------------------------------------------------
# 2) Ordem por escala e por família/configuração
# ---------------------------------------------------------
scale_order = {
    "sf0.25": 1,
    "sf0.5": 2,
    "sf1": 3
}

config_order = {
    "watchitem_g0": 1,
    "watchitem_g2": 2,
    "watchitem_g3": 3,
    "watchitem_g4": 4,
    "watchitem_g5": 5,
    "watchitem_g6": 6,
    "series_g7": 7,
    "series_g8": 8,
    "series_g9": 9
}

family_label_map = {
    "watchitem_g0": "WatchItem rasa",
    "watchitem_g2": "WatchItem rasa",
    "watchitem_g3": "WatchItem rasa",
    "watchitem_g4": "Associativa",
    "watchitem_g5": "Associativa",
    "watchitem_g6": "Associativa",
    "series_g7": "Containment",
    "series_g8": "Containment",
    "series_g9": "Containment"
}

comparison_goal_map = {
    "watchitem_g0": "Baseline raiz WatchItem",
    "watchitem_g2": "Descritor compartilhado",
    "watchitem_g3": "Componente estrutural",
    "watchitem_g4": "Associativa parcial",
    "watchitem_g5": "Associativa com snapshot",
    "watchitem_g6": "Associativa full",
    "series_g7": "Baseline containment",
    "series_g8": "Containment reduzido",
    "series_g9": "Containment completo"
}

wave_map = {
    "sf0.25": "Onda 1 — validação inicial",
    "sf0.5": "Onda 2 — escala intermediária",
    "sf1": "Onda 3 — escala final"
}

# ---------------------------------------------------------
# 3) Construir tabela ordenada
# ---------------------------------------------------------
experiment_execution_plan_df = mongo_experiment_catalog_df.copy()

experiment_execution_plan_df["scale_priority"] = experiment_execution_plan_df["scale_label"].map(scale_order)
experiment_execution_plan_df["config_priority"] = experiment_execution_plan_df["config_name"].map(config_order)

experiment_execution_plan_df["execution_wave"] = experiment_execution_plan_df["scale_label"].map(wave_map)
experiment_execution_plan_df["family_label"] = experiment_execution_plan_df["config_name"].map(family_label_map)
experiment_execution_plan_df["comparison_goal"] = experiment_execution_plan_df["config_name"].map(comparison_goal_map)

experiment_execution_plan_df = experiment_execution_plan_df.sort_values(
    ["scale_priority", "config_priority", "experiment_id"]
).reset_index(drop=True)

experiment_execution_plan_df["execution_order"] = range(1, len(experiment_execution_plan_df) + 1)

# ---------------------------------------------------------
# 4) Coluna textual de prioridade/racional
# ---------------------------------------------------------
def build_execution_rationale(row):
    if row["scale_label"] == "sf0.25":
        return (
            "Executar primeiro para validar carga, modelagem, índices e comportamento "
            "das queries na menor escala."
        )
    elif row["scale_label"] == "sf0.5":
        return (
            "Executar depois da validação inicial para verificar se o comportamento "
            "da configuração se mantém em escala intermediária."
        )
    else:
        return (
            "Executar por último para medir escalabilidade final e confirmar "
            "os trade-offs observados nas escalas menores."
        )

experiment_execution_plan_df["execution_rationale"] = experiment_execution_plan_df.apply(
    build_execution_rationale,
    axis=1
)

# ---------------------------------------------------------
# 5) Tabela-resumo enxuta
# ---------------------------------------------------------
experiment_execution_plan_summary_df = experiment_execution_plan_df[
    [
        "execution_order",
        "execution_wave",
        "experiment_id",
        "config_name",
        "activated_class",
        "family_label",
        "scale_label",
        "selected_root",
        "embedding_variant",
        "comparison_goal",
        "n_primary_queries",
        "n_secondary_affected_queries",
        "n_control_queries",
        "execution_rationale"
    ]
].copy()

print("Tabela-resumo com a ordem recomendada dos experimentos:")
display(experiment_execution_plan_summary_df)

In [ ]:
# =========================================================
# BLOCK V22C — COMPARISON MATRIX BY FAMILY
# prepares comparative analysis of results by:
# - structural family
# - scale factor
# - competing configurations
# =========================================================

import pandas as pd
import itertools

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_experiment_catalog_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V22."
        )

# ---------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def normalize_list(values):
    seen = []
    for v in values:
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

def family_readable_label(family_name):
    mapping = {
        "root_local_family": "WatchItem rasa",
        "associative_family": "Associativa",
        "containment_family": "Containment"
    }
    return mapping.get(family_name, family_name)

def comparison_question_for_family(family_name):
    mapping = {
        "root_local_family": (
            "Quanto cada política rasa melhora consultas centradas na raiz "
            "e qual o custo em queries afetadas?"
        ),
        "associative_family": (
            "Vale mais embutimento parcial, snapshot ou full embedding "
            "na região associativa WatchItem–Role–Person?"
        ),
        "containment_family": (
            "Containment baseline, reduzido ou completo: qual traz o melhor "
            "trade-off na relação Series–Episode?"
        )
    }
    return mapping.get(family_name, "Comparar variantes estruturais da mesma família.")

# ---------------------------------------------------------
# 3) Matriz agregada por família e scale factor
#    Uma linha = um grupo de comparação
# ---------------------------------------------------------
family_scale_rows = []

group_cols = ["benchmark_family", "scale_label"]

for keys, grp in mongo_experiment_catalog_df.groupby(group_cols):
    benchmark_family, scale_label = keys

    grp = grp.sort_values("config_name").reset_index(drop=True)

    config_names = grp["config_name"].tolist()
    experiment_ids = grp["experiment_id"].tolist()
    mongo_db_names = grp["mongo_db_name"].tolist()
    activated_classes = grp["activated_class"].tolist()

    primary_queries = normalize_list(grp["primary_queries"].tolist())
    secondary_queries = normalize_list(grp["secondary_affected_queries"].tolist())
    control_queries = normalize_list(grp["control_queries"].tolist())

    family_scale_rows.append({
        "benchmark_family": benchmark_family,
        "family_label": family_readable_label(benchmark_family),
        "scale_label": scale_label,
        "n_configurations_in_group": len(config_names),
        "config_names": config_names,
        "experiment_ids": experiment_ids,
        "mongo_db_names": mongo_db_names,
        "activated_classes": activated_classes,
        "primary_queries_union": primary_queries,
        "secondary_affected_queries_union": secondary_queries,
        "control_queries_union": control_queries,
        "n_primary_queries_union": len(primary_queries),
        "n_secondary_affected_queries_union": len(secondary_queries),
        "n_control_queries_union": len(control_queries),
        "comparison_question": comparison_question_for_family(benchmark_family)
    })

family_scale_comparison_df = pd.DataFrame(family_scale_rows).sort_values(
    ["scale_label", "benchmark_family"]
).reset_index(drop=True)

print("Matriz de comparação por família e scale factor:")
display(family_scale_comparison_df)

# ---------------------------------------------------------
# 4) Comparações par-a-par dentro de cada família e escala
#    Útil para análises mais finas de trade-off
# ---------------------------------------------------------
pairwise_rows = []

for _, row in family_scale_comparison_df.iterrows():
    benchmark_family = row["benchmark_family"]
    family_label = row["family_label"]
    scale_label = row["scale_label"]

    config_names = as_list(row["config_names"])
    experiment_ids = as_list(row["experiment_ids"])
    mongo_db_names = as_list(row["mongo_db_names"])
    activated_classes = as_list(row["activated_classes"])

    config_records = list(zip(config_names, experiment_ids, mongo_db_names, activated_classes))

    for (cfg_a, exp_a, db_a, cls_a), (cfg_b, exp_b, db_b, cls_b) in itertools.combinations(config_records, 2):
        pairwise_rows.append({
            "benchmark_family": benchmark_family,
            "family_label": family_label,
            "scale_label": scale_label,
            "config_a": cfg_a,
            "config_b": cfg_b,
            "experiment_id_a": exp_a,
            "experiment_id_b": exp_b,
            "mongo_db_a": db_a,
            "mongo_db_b": db_b,
            "activated_class_a": cls_a,
            "activated_class_b": cls_b,
            "comparison_goal": (
                f"Comparar {cfg_a} versus {cfg_b} dentro da família "
                f"{family_label} em {scale_label}."
            )
        })

pairwise_family_comparison_df = pd.DataFrame(pairwise_rows)

print("\nComparações par-a-par dentro de cada família e scale factor:")
display(pairwise_family_comparison_df)

# ---------------------------------------------------------
# 5) Versão amigável para visualização
# ---------------------------------------------------------
family_scale_comparison_view_df = family_scale_comparison_df.copy()

for col in [
    "config_names",
    "experiment_ids",
    "mongo_db_names",
    "activated_classes",
    "primary_queries_union",
    "secondary_affected_queries_union",
    "control_queries_union"
]:
    family_scale_comparison_view_df[col] = family_scale_comparison_view_df[col].apply(list_to_str)

print("\nVersão amigável da matriz de comparação por família:")
display(family_scale_comparison_view_df)

# ---------------------------------------------------------
# 6) Ordem sugerida de comparação analítica
#    dentro de cada scale factor:
#    1) família root_local
#    2) família associativa
#    3) família containment
# ---------------------------------------------------------
family_order = {
    "root_local_family": 1,
    "associative_family": 2,
    "containment_family": 3
}

scale_order = {
    "sf0.25": 1,
    "sf0.5": 2,
    "sf1": 3
}

comparison_execution_plan_df = family_scale_comparison_df.copy()
comparison_execution_plan_df["scale_priority"] = comparison_execution_plan_df["scale_label"].map(scale_order)
comparison_execution_plan_df["family_priority"] = comparison_execution_plan_df["benchmark_family"].map(family_order)

comparison_execution_plan_df = comparison_execution_plan_df.sort_values(
    ["scale_priority", "family_priority"]
).reset_index(drop=True)

comparison_execution_plan_df["comparison_execution_order"] = range(
    1, len(comparison_execution_plan_df) + 1
)

comparison_execution_plan_summary_df = comparison_execution_plan_df[
    [
        "comparison_execution_order",
        "scale_label",
        "family_label",
        "n_configurations_in_group",
        "n_primary_queries_union",
        "n_secondary_affected_queries_union",
        "n_control_queries_union",
        "comparison_question"
    ]
].copy()

print("\nPlano sugerido de comparação dos resultados por família:")
display(comparison_execution_plan_summary_df)

In [ ]:
# =========================================================
# BLOCK V22D — BENCHMARK EXECUTION TEMPLATE
# one row per:
# experimento × query × query_group × run_phase × repetition
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_experiment_catalog_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first o bloco V22."
        )

# ---------------------------------------------------------
# 2) Parâmetros globais do benchmark
# ---------------------------------------------------------
BENCHMARK_REPETITIONS = 10

# Fases sugeridas pela metodologia:
# - cold: primeira execução / ambiente frio
# - hot: execução subsequente / cache aquecido
BENCHMARK_PHASES = [
    {"run_phase": "cold", "phase_order": 1},
    {"run_phase": "hot",  "phase_order": 2},
]

benchmark_phases_df = pd.DataFrame(BENCHMARK_PHASES)

# ---------------------------------------------------------
# 3) Helpers
# ---------------------------------------------------------
def as_list(x):
    return x if isinstance(x, list) else []

def normalize_list(values):
    seen = []
    for v in values:
        if isinstance(v, list):
            for x in v:
                if x not in seen:
                    seen.append(x)
        else:
            if v not in seen:
                seen.append(v)
    return seen

def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

# ---------------------------------------------------------
# 4) Expandir o catálogo de experimentos em linhas de execução
# ---------------------------------------------------------
benchmark_template_rows = []

for _, exp_row in mongo_experiment_catalog_df.iterrows():
    experiment_id = exp_row["experiment_id"]

    primary_queries = normalize_list(exp_row.get("primary_queries", []))
    secondary_queries = normalize_list(exp_row.get("secondary_affected_queries", []))
    control_queries = normalize_list(exp_row.get("control_queries", []))

    query_groups = [
        ("primary", primary_queries),
        ("secondary_affected", secondary_queries),
        ("control", control_queries),
    ]

    for query_group, query_list in query_groups:
        for query_name in query_list:
            for _, phase_row in benchmark_phases_df.iterrows():
                run_phase = phase_row["run_phase"]
                phase_order = phase_row["phase_order"]

                for repetition in range(1, BENCHMARK_REPETITIONS + 1):
                    benchmark_template_rows.append({
                        # identidade da execução
                        "benchmark_run_id": (
                            f"{experiment_id}__{query_group}__{query_name}__"
                            f"{run_phase}__r{repetition:02d}"
                        ),
                        "experiment_id": experiment_id,
                        "config_name": exp_row["config_name"],
                        "activated_class": exp_row["activated_class"],
                        "benchmark_family": exp_row["benchmark_family"],

                        # escala
                        "scale_factor": exp_row["scale_factor"],
                        "scale_label": exp_row["scale_label"],

                        # banco / configuração física
                        "mongo_db_name": exp_row["mongo_db_name"],
                        "selected_root": exp_row["selected_root"],
                        "primary_collection": exp_row["primary_collection"],
                        "embedding_depth": exp_row["embedding_depth"],
                        "embedding_variant": exp_row["embedding_variant"],
                        "embedded_entities": exp_row["embedded_entities"],
                        "snapshot_entities": exp_row["snapshot_entities"],
                        "referenced_entities": exp_row["referenced_entities"],

                        # query executada
                        "query_name": query_name,
                        "query_group": query_group,   # primary / secondary_affected / control
                        "run_phase": run_phase,       # cold / hot
                        "phase_order": phase_order,
                        "repetition": repetition,

                        # status operacional
                        "execution_status": "planned",
                        "error_message": None,

                        # métricas brutas a preencher depois
                        "start_ts": None,
                        "end_ts": None,
                        "latency_ms": None,
                        "success": None,
                        "documents_returned": None,
                        "documents_written": None,
                        "bytes_read_estimate": None,
                        "bytes_written_estimate": None,

                        # metadados de interpretação
                        "experiment_goal": exp_row["experiment_goal"]
                    })

benchmark_execution_template_df = pd.DataFrame(benchmark_template_rows)

# ---------------------------------------------------------
# 5) Ordenação
# ---------------------------------------------------------
query_group_order = {
    "primary": 1,
    "secondary_affected": 2,
    "control": 3
}

benchmark_execution_template_df["query_group_order"] = (
    benchmark_execution_template_df["query_group"].map(query_group_order)
)

benchmark_execution_template_df = benchmark_execution_template_df.sort_values(
    [
        "scale_factor",
        "config_name",
        "query_group_order",
        "query_name",
        "phase_order",
        "repetition"
    ]
).reset_index(drop=True)

# ---------------------------------------------------------
# 6) Visões resumidas
# ---------------------------------------------------------
benchmark_execution_plan_summary_df = (
    benchmark_execution_template_df
    .groupby(
        [
            "experiment_id",
            "config_name",
            "activated_class",
            "benchmark_family",
            "scale_label",
            "query_group",
            "run_phase"
        ],
        as_index=False
    )
    .agg(
        n_rows=("benchmark_run_id", "count"),
        n_distinct_queries=("query_name", "nunique"),
        min_repetition=("repetition", "min"),
        max_repetition=("repetition", "max")
    )
    .sort_values(
        ["scale_label", "config_name", "query_group", "run_phase"]
    )
    .reset_index(drop=True)
)

benchmark_query_inventory_df = (
    benchmark_execution_template_df
    .groupby(
        [
            "experiment_id",
            "query_group"
        ],
        as_index=False
    )
    .agg(
        query_names=("query_name", lambda s: sorted(s.unique().tolist())),
        n_distinct_queries=("query_name", "nunique")
    )
    .sort_values(["experiment_id", "query_group"])
    .reset_index(drop=True)
)

benchmark_query_inventory_view_df = benchmark_query_inventory_df.copy()
benchmark_query_inventory_view_df["query_names"] = benchmark_query_inventory_view_df["query_names"].apply(list_to_str)

# ---------------------------------------------------------
# 7) Tabela de métricas agregadas (vazia, pronta para uso depois)
#    Esta tabela é o molde do resultado consolidado do benchmark.
# ---------------------------------------------------------
benchmark_result_aggregate_template_df = pd.DataFrame(columns=[
    "experiment_id",
    "config_name",
    "activated_class",
    "benchmark_family",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "n_runs",
    "n_success_runs",
    "avg_latency_ms",
    "median_latency_ms",
    "p95_latency_ms",
    "p99_latency_ms",
    "min_latency_ms",
    "max_latency_ms",
    "std_latency_ms",
    "avg_documents_returned",
    "avg_documents_written",
    "avg_bytes_read_estimate",
    "avg_bytes_written_estimate"
])

# ---------------------------------------------------------
# 8) Display
# ---------------------------------------------------------
print("Template detalhado da execução do benchmark:")
display(benchmark_execution_template_df)

print("\nSummary do plano de execução por experimento / grupo / fase:")
display(benchmark_execution_plan_summary_df)

print("\nInventário de queries por experimento:")
display(benchmark_query_inventory_view_df)

print("\nTemplate da tabela agregada de resultados:")
display(benchmark_result_aggregate_template_df)

In [ ]:
#salvar todos os resultafos das matrizes geradas em uma pasta
import os
import pandas as pd

pasta_saida = "/home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/backup_variaveis"
os.makedirs(pasta_saida, exist_ok=True)

for nome, valor in list(globals().items()):
    if isinstance(valor, pd.DataFrame):
        caminho = os.path.join(pasta_saida, f"{nome}.csv")
        valor.to_csv(caminho, index=False)
        print(f"Salvo: {caminho}")

## Part B — Optional MongoDB validation / focal benchmark blocks

These blocks are **optional** and are meant for manual validation of a specific MongoDB payload / experiment. They depend on a running MongoDB instance and, in some cases, on variables created outside the core notebook. The original `V26` block was omitted here because it depends on `V25`, which is not present in the provided notebook.

### Optional environment notes

- Example SSH tunnel from the original notebook:

```bash
ssh -N -L 57017:127.0.0.1:27018 hudson@150.162.57.138
```

- Install PyMongo if needed:

```bash
pip install pymongo
```

- Optional connectivity checks for PostgreSQL/Cassandra/MongoDB existed in the original notebook and can be run separately if needed.

In [ ]:
# =========================================================
# BLOCK V27 — MONGODB CONNECTION AND UTILITIES
# =========================================================

from pathlib import Path

try:
    from pymongo import MongoClient
    HAS_PYMONGO = True
except ImportError:
    HAS_PYMONGO = False
    MongoClient = None


# ---------------------------------------------------------
# 1) Função para abrir conexão com MongoDB
# ---------------------------------------------------------
def mongo_connect_basic(host="127.0.0.1", port=57017, username="mongo", password="mongo", auth_source="admin"):
    """
    Abre conexão com MongoDB.

    Ajuste host/port se você estiver usando:
    - Docker com porta diferente
    - túnel SSH
    - autenticação
    """
    if not HAS_PYMONGO:
        raise ImportError(
            "pymongo não está instalado. Rode: pip install pymongo"
        )

    if username and password:
        client = MongoClient(
            host=host,
            port=port,
            username=username,
            password=password,
            authSource=auth_source,
            serverSelectionTimeoutMS=5000
        )
    else:
        client = MongoClient(
            host=host,
            port=port,
            serverSelectionTimeoutMS=5000
        )

    # força teste de conexão
    client.admin.command("ping")
    return client


# ---------------------------------------------------------
# 2) Função para apagar um database inteiro, se existir
# ---------------------------------------------------------
def drop_mongo_database_if_exists(client, db_name: str):
    """
    Remove o database inteiro, se ele existir.
    Útil para recomeçar o experimento limpo.
    """
    existing_dbs = client.list_database_names()

    if db_name in existing_dbs:
        client.drop_database(db_name)
        print(f"Database removido: {db_name}")
    else:
        print(f"Database ainda não existia: {db_name}")


# ---------------------------------------------------------
# 3) Função para contar documentos de todas as coleções
# ---------------------------------------------------------
def verify_mongo_collection_counts(db, collection_names):
    """
    Retorna um DataFrame com a contagem de documentos por coleção.
    """
    rows = []

    for collection_name in collection_names:
        count = db[collection_name].count_documents({})
        rows.append({
            "collection_name": collection_name,
            "document_count": count
        })

    return pd.DataFrame(rows).sort_values("collection_name").reset_index(drop=True)

In [ ]:
# =========================================================
# BLOCK V28 — LOAD THE M1 PAYLOAD INTO MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_payload_m1",
    "mongo_connect_basic",
    "drop_mongo_database_if_exists",
    "verify_mongo_collection_counts"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se sua porta do MongoDB não for 27017
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Conectar
# ---------------------------------------------------------
mongo_client = mongo_connect_basic(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE
)

print("Conexão com MongoDB: OK")

# ---------------------------------------------------------
# 4) Select database do experimento
# ---------------------------------------------------------
target_db_name = mongo_payload_m1.mongo_db_name
print("Database alvo:", target_db_name)

# ---------------------------------------------------------
# 5) Apagar database anterior, se existir
# Isso garante repetibilidade do experimento.
# ---------------------------------------------------------
drop_mongo_database_if_exists(mongo_client, target_db_name)

# ---------------------------------------------------------
# 6) Create database e carregar as coleções
# ---------------------------------------------------------
mongo_db = mongo_client[target_db_name]

for collection_name, documents in mongo_payload_m1.collections.items():
    if len(documents) > 0:
        mongo_db[collection_name].insert_many(documents)
        print(f"Coleção carregada: {collection_name} ({len(documents)} documentos)")
    else:
        print(f"Coleção vazia ignorada: {collection_name}")

print("\nCarga MongoDB finalizada.")

In [ ]:
# =========================================================
# BLOCK V29 — CREATE INDEXES FOR M1-FULL
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Índices da coleção principal watchitems
# M1 foi pensado principalmente para Q2 e Q4
# ---------------------------------------------------------

# Busca por título (Q2)
mongo_db["watchitems"].create_index([("title", 1)])

# Filtro por gênero, tipo e ano (Q4)
mongo_db["watchitems"].create_index([
    ("genre_id", 1),
    ("item_type", 1),
    ("release_year", 1)
])

# Campo auxiliar de subtipo
mongo_db["watchitems"].create_index([("document_subtype", 1)])

# ---------------------------------------------------------
# 3) Índices das demais coleções do domínio
# Isso ajuda a manter o banco full mais realista e também
# prepara futuras consultas/benchmarks.
# ---------------------------------------------------------
mongo_db["users"].create_index([("user_id", 1)], unique=True)
mongo_db["persons"].create_index([("person_id", 1)], unique=True)
mongo_db["genres"].create_index([("genre_id", 1)], unique=True)
mongo_db["movies"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["series"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["episodes"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["roles"].create_index([("role_id", 1)], unique=True)
mongo_db["rates"].create_index([("rate_id", 1)], unique=True)

# Índices auxiliares úteis
mongo_db["roles"].create_index([
    ("watchitem_id", 1),
    ("role_type", 1),
    ("person_id", 1)
])

mongo_db["rates"].create_index([
    ("user_id", 1),
    ("watchitem_id", 1)
])

print("Índices do M1-full criados com sucesso.")

In [ ]:
# =========================================================
# BLOCK V30 — VALIDATE THE M1 LOAD
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Verificar contagem por coleção
# ---------------------------------------------------------
m1_counts_df = verify_mongo_collection_counts(
    mongo_db,
    collection_names=mongo_payload_m1.collection_names()
)

print("Contagens no MongoDB para o experimento M1:")
display(m1_counts_df)

# ---------------------------------------------------------
# 3) Verificar um documento de exemplo
# ---------------------------------------------------------
sample_watchitem_doc = mongo_db["watchitems"].find_one({}, {"_id": 0})

print("Exemplo de documento salvo no MongoDB (watchitems):")
display(pd.DataFrame([sample_watchitem_doc]))

In [ ]:
# =========================================================
# BLOCK V31 — TEST QUERY FOR M1
# =========================================================

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "The object 'mongo_db' is not defined. Run first o bloco V28."
    )

# ---------------------------------------------------------
# 2) Teste simples no estilo Q4
# Escolhe um documento qualquer e usa seus próprios valores
# ---------------------------------------------------------
sample_for_q4 = mongo_db["watchitems"].find_one(
    {},
    {
        "_id": 0,
        "genre_id": 1,
        "item_type": 1,
        "release_year": 1
    }
)

if sample_for_q4 is None:
    raise ValueError("Nenhum documento encontrado em watchitems.")

genre_id = sample_for_q4["genre_id"]
item_type = sample_for_q4["item_type"]
center_year = sample_for_q4["release_year"]

year_low = center_year - 10
year_high = center_year + 10

query = {
    "genre_id": genre_id,
    "item_type": item_type,
    "release_year": {
        "$gte": year_low,
        "$lte": year_high
    }
}

q4_result = list(
    mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )
)

print("Parâmetros usados no teste tipo Q4:")
print({
    "genre_id": genre_id,
    "item_type": item_type,
    "year_low": year_low,
    "year_high": year_high
})

print(f"\nNúmero de documentos retornados: {len(q4_result)}")

if len(q4_result) > 0:
    display(pd.DataFrame(q4_result[:10]))

In [ ]:
# =========================================================
# BLOCK V32 — BENCHMARK CONNECTION FOR M1
# =========================================================

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "mongo_connect_basic",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se seu Mongo estiver em outra porta
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Funções utilitárias para benchmark
# ---------------------------------------------------------
def open_mongo_benchmark_connection(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE,
    db_name=None
):
    """
    Abre conexão Mongo e retorna:
    - client
    - db
    """
    client = mongo_connect_basic(
        host=host,
        port=port,
        username=username,
        password=password,
        auth_source=auth_source
    )

    if db_name is None:
        raise ValueError("db_name precisa ser informado.")

    db = client[db_name]
    return client, db


def close_mongo_benchmark_connection(client):
    """
    Fecha a conexão Mongo de forma limpa.
    """
    try:
        if client is not None:
            client.close()
    except Exception:
        pass

# ---------------------------------------------------------
# 4) Abrir uma conexão de teste
# ---------------------------------------------------------
mongo_bench_client, mongo_bench_db = open_mongo_benchmark_connection(
    db_name=active_experiment_row["mongo_db_name"]
)

print("Conexão de benchmark MongoDB aberta com sucesso.")
print("Database:", active_experiment_row["mongo_db_name"])

In [ ]:
# =========================================================
# BLOCK V33 — MONGODB QUERIES FOR M1
# =========================================================

# ---------------------------------------------------------
# Q2 — SimpleSearch no M1
# Search by title directly in the watchitems collection
# ---------------------------------------------------------
def run_q2_simple_search_m1_mongodb(mongo_db, title: str):
    cursor = mongo_db["watchitems"].find(
        {"title": str(title)},
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1,
            "movie_data": 1,
            "series_data": 1,
            "episode_data": 1
        }
    )
    return list(cursor)


# ---------------------------------------------------------
# Q4 — Recommendation no M1
# Filtra diretamente em watchitems por:
# - genre_id
# - item_type
# - release_year em faixa
# ---------------------------------------------------------
def run_q4_recommendation_m1_mongodb(
    mongo_db,
    genre_id: int,
    item_type: str,
    year_low: int,
    year_high: int
):
    query = {
        "genre_id": int(genre_id),
        "item_type": str(item_type),
        "release_year": {
            "$gte": int(year_low),
            "$lte": int(year_high)
        }
    }

    cursor = mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )

    return list(cursor)


# ---------------------------------------------------------
# Teste rápido das funções
# ---------------------------------------------------------
sample_title = mongo_bench_db["watchitems"].find_one({}, {"_id": 0, "title": 1})["title"]

q2_test_result = run_q2_simple_search_m1_mongodb(
    mongo_bench_db,
    title=sample_title
)

print("Teste rápido da Q2 no M1:")
print("Título usado:", sample_title)
print("Número de documentos retornados:", len(q2_test_result))

sample_q4 = mongo_bench_db["watchitems"].find_one(
    {},
    {"_id": 0, "genre_id": 1, "item_type": 1, "release_year": 1}
)

q4_test_result = run_q4_recommendation_m1_mongodb(
    mongo_bench_db,
    genre_id=sample_q4["genre_id"],
    item_type=sample_q4["item_type"],
    year_low=sample_q4["release_year"] - 10,
    year_high=sample_q4["release_year"] + 10
)

print("\nTeste rápido da Q4 no M1:")
print("Parâmetros usados:", sample_q4)
print("Número de documentos retornados:", len(q4_test_result))

In [ ]:
# =========================================================
# BLOCK V34B — MINI-BENCHMARK COM VALIDAÇÃO AUTOMÁTICA
# =========================================================

import time
import random
import pandas as pd

def run_m1_focal_benchmark_validated(
    mongo_db,
    parameter_pool,
    cold_repetitions=3,
    hot_repetitions=20,
    seed=42
):
    rng = random.Random(seed)
    results = []

    def append_result(query_name, phase, run_id, start_time, success, error_message):
        latency_ms = (time.perf_counter() - start_time) * 1000.0
        results.append({
            "experiment_id": active_experiment_row["experiment_id"],
            "mongo_db_name": active_experiment_row["mongo_db_name"],
            "query_name": query_name,
            "benchmark_phase": phase,
            "run_id": run_id,
            "latency_ms": latency_ms,
            "success": success,
            "error_message": error_message
        })

    # -----------------------------
    # COLD RUNS — Q2
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q2_simple_search_m1_mongodb(local_db, title)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q2_SimpleSearch", "cold", run_id, start, success, error_message)

    # -----------------------------
    # COLD RUNS — Q4
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q4_recommendation_m1_mongodb(
                local_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q4_Recommendation", "cold", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q2
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q2_simple_search_m1_mongodb(mongo_db, title)
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q2_SimpleSearch", "hot", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q4
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q4_recommendation_m1_mongodb(
                mongo_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q4_Recommendation", "hot", run_id, start, success, error_message)

    benchmark_df = pd.DataFrame(results)

    # -----------------------------
    # VALIDAÇÃO AUTOMÁTICA
    # -----------------------------
    expected_counts = {
        ("Q2_SimpleSearch", "cold"): cold_repetitions,
        ("Q2_SimpleSearch", "hot"): hot_repetitions,
        ("Q4_Recommendation", "cold"): cold_repetitions,
        ("Q4_Recommendation", "hot"): hot_repetitions,
    }

    actual_counts = (
        benchmark_df
        .groupby(["query_name", "benchmark_phase"])
        .size()
        .to_dict()
    )

    for key, expected_value in expected_counts.items():
        actual_value = actual_counts.get(key, 0)
        if actual_value != expected_value:
            raise ValueError(
                f"Contagem inesperada para {key}: esperado={expected_value}, obtido={actual_value}"
            )

    return benchmark_df

In [ ]:
# =========================================================
# BLOCK V34A — DIAGNOSTICS AND CORRECT DISPLAY FOR THE MINI-BENCHMARK
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Basic check
# ---------------------------------------------------------
if "m1_benchmark_df" not in globals():
    raise NameError(
        "O DataFrame 'm1_benchmark_df' is not defined. "
        "Rode primeiro o bloco do mini-benchmark."
    )

# ---------------------------------------------------------
# 2) Sort explicitamente o resultado
# Isso ajuda a visualizar tudo de forma consistente
# ---------------------------------------------------------
m1_benchmark_df = m1_benchmark_df.sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 3) Contagem de execuções por query e fase
# Esse é o teste mais importante para saber se Q4 hot existe
# ---------------------------------------------------------
m1_counts_by_query_phase_df = (
    m1_benchmark_df
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        n_runs=("run_id", "count"),
        n_success=("success", lambda s: int(s.sum())),
        n_failures=("success", lambda s: int((~s).sum()))
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Contagem de execuções por query e fase:")
display(m1_counts_by_query_phase_df)

# ---------------------------------------------------------
# 4) Mostrar explicitamente as linhas de Q4 hot
# ---------------------------------------------------------
m1_q4_hot_df = m1_benchmark_df[
    (m1_benchmark_df["query_name"] == "Q4_Recommendation") &
    (m1_benchmark_df["benchmark_phase"] == "hot")
].copy()

print("Linhas de Q4 hot:")
display(m1_q4_hot_df)

# ---------------------------------------------------------
# 5) Summary estatístico simples por query e fase
# ---------------------------------------------------------
m1_summary_df = (
    m1_benchmark_df[m1_benchmark_df["success"] == True]
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        runs=("latency_ms", "count"),
        avg_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max")
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Summary estatístico do mini-benchmark:")
display(m1_summary_df)

# ---------------------------------------------------------
# 6) Mostrar tudo, se quiser inspecionar linha a linha
# ---------------------------------------------------------
print("Resultados brutos completos do mini-benchmark:")
display(m1_benchmark_df)

# ---------------------------------------------------------
# 7) Falhas, se existirem
# ---------------------------------------------------------
m1_failures_df = m1_benchmark_df[m1_benchmark_df["success"] == False].copy()

print("Falhas, se existirem:")
display(m1_failures_df)

In [ ]:
# =========================================================
# BLOCK V35 — EXPORT BENCHMARK RESULTS PER EXPERIMENT
# =========================================================

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------
required_names = [
    "m1_benchmark_df",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' is not defined. Run first os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função para resumir estatísticas básicas
# ---------------------------------------------------------
def summarize_benchmark_basic(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um resumo simples por query e fase.
    """
    if result_df.empty:
        return pd.DataFrame()

    ok_df = result_df[result_df["success"] == True].copy()

    if ok_df.empty:
        return pd.DataFrame()

    summary_df = (
        ok_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            runs=("latency_ms", "count"),
            avg_ms=("latency_ms", "mean"),
            median_ms=("latency_ms", "median"),
            min_ms=("latency_ms", "min"),
            max_ms=("latency_ms", "max"),
            std_ms=("latency_ms", "std")
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    summary_df["std_ms"] = summary_df["std_ms"].fillna(0.0)
    return summary_df


# ---------------------------------------------------------
# 3) Função para contar execuções por query e fase
# ---------------------------------------------------------
def build_counts_by_query_phase(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Conta quantas execuções ocorreram por query e fase,
    incluindo sucessos e falhas.
    """
    if result_df.empty:
        return pd.DataFrame()

    counts_df = (
        result_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            n_runs=("run_id", "count"),
            n_success=("success", lambda s: int(s.sum())),
            n_failures=("success", lambda s: int((~s).sum()))
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    return counts_df


# ---------------------------------------------------------
# 4) Define diretório do experimento
# Cada experimento ganha sua própria pasta
# ---------------------------------------------------------
experiment_id = active_experiment_row["experiment_id"]

experiment_results_dir = Path("exports/mongodb_benchmarks") / experiment_id
experiment_results_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 5) Preparar DataFrames para exportação
# ---------------------------------------------------------
benchmark_raw_df = m1_benchmark_df.copy().sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

benchmark_summary_df = summarize_benchmark_basic(benchmark_raw_df)

benchmark_failures_df = benchmark_raw_df[
    benchmark_raw_df["success"] == False
].copy().reset_index(drop=True)

benchmark_counts_df = build_counts_by_query_phase(benchmark_raw_df)

# ---------------------------------------------------------
# 6) Define nomes dos arquivos com prefixo do experimento
# ---------------------------------------------------------
raw_csv_path = experiment_results_dir / f"{experiment_id}_raw.csv"
summary_csv_path = experiment_results_dir / f"{experiment_id}_summary.csv"
failures_csv_path = experiment_results_dir / f"{experiment_id}_failures.csv"
counts_csv_path = experiment_results_dir / f"{experiment_id}_counts_by_query_phase.csv"

# ---------------------------------------------------------
# 7) Salvar os CSVs
# ---------------------------------------------------------
benchmark_raw_df.to_csv(raw_csv_path, index=False)
benchmark_summary_df.to_csv(summary_csv_path, index=False)
benchmark_failures_df.to_csv(failures_csv_path, index=False)
benchmark_counts_df.to_csv(counts_csv_path, index=False)

# ---------------------------------------------------------
# 8) Mostrar os caminhos gerados
# ---------------------------------------------------------
print("Arquivos CSV do benchmark exportados com sucesso:")
print(raw_csv_path)
print(summary_csv_path)
print(failures_csv_path)
print(counts_csv_path)

# ---------------------------------------------------------
# 9) Mostrar prévia dos dados exportados
# ---------------------------------------------------------
print("\nSummary estatístico exportado:")
display(benchmark_summary_df)

print("Contagem por query e fase exportada:")
display(benchmark_counts_df)

print("Falhas exportadas:")
display(benchmark_failures_df)